# Countries domain — Wikidata multi-hop benchmark generator (v44 rich events / Olympics / wars / Nobel high-recall queue)

This patched notebook generates RU/EN countries-domain benchmark examples with complete Wikidata gold answers and the same JSONL schema used by the curated cinema/people domains.

Main design points:

- deterministic candidate queues instead of random infinite retries;
- incremental JSONL append after every accepted record;
- clean user-facing English constraints, while QIDs/property paths stay in `gold_collection_meta.bridge_meta`;
- full top-level schema compatible with `cinema.jsonl` / `people.jsonl`;
- all golds are collected directly from WDQS, with `ask_validator_sparql` for every record;
- L3/L4/L5 use real multi-hop country patterns: capitals, neighbouring countries, current organization membership, UNESCO World Heritage Sites, and large cities;
- no intentionally impossible questions and no "if fewer than 5" prompts.


v36/v37 fixes:
- uses exact Wikidata class `sovereign state` (`P31 = Q3624078`) instead of broad `country` subclass traversal, so micronations/dependent territories/constituent countries do not leak into golds;
- progress bars now count accepted records, while tried candidates are shown in the postfix.


v37 path cleanup:
- writes clean domain output to `out_wikidata_benchmark/domain_outputs/countries.jsonl`;
- audit and checkpoint are `countries_audit.json` and `countries_checkpoint.json`;
- rejects/resets old non-strict rows if they used the broad `country` filter instead of exact `sovereign state`.



v38/v39 quality fixes:
- removes unsafe negative membership generation;
- adds gold-overlap deduplication constants to prevent duplicate/subset answer sets;
- filters known Wikidata false-positive official-language golds;
- cleans RU/EN query text for membership/population templates.


v44 additions:
- adds richer country-domain bridges: Nobel laureate citizens, Olympic medalist citizens, hosted global events, participation in major conflicts, capitals located on rivers, highest points;
- expands L4/L5 with broader positive-only multi-hop templates so hard levels can be generated with запас;
- raises L4/L5 targets and candidate queues for a fresh overgeneration run;
- keeps constraints clean and human-readable; QIDs/property paths stay in bridge metadata;
- keeps unsafe negative membership disabled.


In [1]:
# Load common helpers only if this domain notebook is run standalone.
# This avoids `%run ./00_common_helpers.ipynb`, which requires nbformat.
from pathlib import Path as _Path

if "BenchmarkExample" not in globals():
    helper_path = _Path("common_helpers.py")
    if not helper_path.exists():
        raise FileNotFoundError("common_helpers.py must be in the same directory as 12_countries.ipynb")
    exec(helper_path.read_text(encoding="utf-8"), globals())

print("✅ common helpers loaded")


✅ Patched: WikidataClient.sparql_select (robust) + load_or_build_pool (safe)
✅ Patched: select_items_with_* используют ru/en fallback + repair_pool_labels чинит QID вместо label
✅ common helpers loaded


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from __future__ import annotations

import json
import random
import re
import time
import datetime as dt
from collections import Counter, defaultdict
from dataclasses import asdict, is_dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

try:
    from tqdm.auto import tqdm
except Exception:
    import sys
    !{sys.executable} -m pip install tqdm
    from tqdm.auto import tqdm

COUNTRIES_DOMAIN = "countries"
COUNTRIES_OUTPUT_DIR = Path("out_wikidata_benchmark/domain_outputs")
COUNTRIES_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# One clean output for the patched generator, using the same simple naming style as other domains.
# If this file already contains old broad-country rows, the generator will drop them during resume.
COUNTRIES_OUTPUT_PATH = COUNTRIES_OUTPUT_DIR / "countries.jsonl"
COUNTRIES_AUDIT_PATH = COUNTRIES_OUTPUT_DIR / "countries_audit.json"
COUNTRIES_CHECKPOINT_PATH = COUNTRIES_OUTPUT_DIR / "countries_checkpoint.json"

# Optional reference JSONLs are used only for duplicate avoidance. They are not copied into the output.
COUNTRIES_REFERENCE_JSONL_PATHS = [
    Path("countries.jsonl"),
    Path("countries_final_curated.jsonl"),
    COUNTRIES_OUTPUT_DIR / "countries.jsonl",
    COUNTRIES_OUTPUT_DIR / "countries_final_curated.jsonl",
    COUNTRIES_OUTPUT_DIR / "countries_queue.jsonl",
]

# IMPORTANT for generation speed/quality:
# Keep this False while generating a fresh countries.jsonl. Previous partial runs are often
# highly overlapping and can cause the generator to reject almost every candidate after an
# expensive WDQS call. Reference files are useful only for a final merge/curation pass.
COUNTRIES_USE_REFERENCE_FILES_FOR_DEDUP = False


# Target 120 records. Keep many hard/multi-hop records while preserving easier levels for level balance.
COUNTRIES_TARGET_PER_LEVEL = {
    # Countries is a small domain. L2 naturally saturates around 15-16 good non-duplicate rows,
    # so keep L2 realistic and spend the budget on harder L4/L5 multi-hop rows.
    "L1": 15,
    "L2": 15,
    "L3": 25,
    "L4": 30,
    "L5": 30,
}

COUNTRIES_RANDOM_SEED = 12041
COUNTRIES_WDQS_LIMIT = 501
COUNTRIES_MAX_CANDIDATES_PER_LEVEL = 3000
COUNTRIES_ACCEPT_MIN_GOLD = {"L1": 8, "L2": 6, "L3": 5, "L4": 5, "L5": 5}
COUNTRIES_ACCEPT_MAX_GOLD = {"L1": 90, "L2": 70, "L3": 70, "L4": 70, "L5": 70}
COUNTRIES_MIN_RU_LABEL_SHARE = 0.60
COUNTRIES_MAX_SAME_TEMPLATE_PER_LEVEL = {"L1": 7, "L2": 8, "L3": 10, "L4": 18, "L5": 20}
COUNTRIES_MAX_SAME_PRIMARY_CONSTRAINT_PER_LEVEL = {"L1": 8, "L2": 10, "L3": 12, "L4": 18, "L5": 20}

# Gold-answer overlap deduplication. This catches exact/near-exact answer-set duplicates.
# Important: for countries, L2 is often a legitimate subset of L1. Therefore containment
# is now used only for near-same-size answer sets, not for ordinary narrower questions.
COUNTRIES_DEDUP_GOLD_JACCARD_THRESHOLD = 0.96
COUNTRIES_DEDUP_GOLD_CONTAINMENT_THRESHOLD = 0.995
COUNTRIES_DEDUP_GOLD_SIZE_RATIO_THRESHOLD = 0.94

# Do not spend the whole queue on a saturated hard level. This kicks in only after
# enough accepted examples exist and a long no-accept streak starts.
COUNTRIES_STAGNATION_LIMIT_BY_LEVEL = {"L1": 120, "L2": 300, "L3": 300, "L4": 350, "L5": 400}
COUNTRIES_MIN_ACCEPTED_BEFORE_STAGNATION = {"L1": 12, "L2": 12, "L3": 20, "L4": 22, "L5": 22}

# Countries are a small domain, but some bridge patterns can still be slow on WDQS.
try:
    wd.timeout = 25
    wd.max_retries = 1
except Exception:
    pass

print("output:", COUNTRIES_OUTPUT_PATH.resolve())


output: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/domain_outputs/countries.jsonl


In [3]:
# ============================================================
# Stable country-domain entities, labels, and property-path metadata
# ============================================================

Q_COUNTRY = "Q6256"            # country (broad; kept only for metadata/comments)
Q_SOVEREIGN_STATE = "Q3624078"   # sovereign state; use exact P31 to avoid micronations/dependent territories
Q_CITY = "Q515"                # city
Q_UNESCO_WHS = "Q9259"         # UNESCO World Heritage Site

COUNTRY_STATUS_CURRENT = "current"

CONTINENTS: Dict[str, Dict[str, str]] = {
    "Europe": {"qid": "Q46", "ru": "Европа", "en": "Europe"},
    "Asia": {"qid": "Q48", "ru": "Азия", "en": "Asia"},
    "Africa": {"qid": "Q15", "ru": "Африка", "en": "Africa"},
    "North America": {"qid": "Q49", "ru": "Северная Америка", "en": "North America"},
    "South America": {"qid": "Q18", "ru": "Южная Америка", "en": "South America"},
    "Oceania": {"qid": "Q55643", "ru": "Океания", "en": "Oceania"},
}

LANGUAGES: Dict[str, Dict[str, str]] = {
    "English": {"qid": "Q1860", "ru": "английский язык", "en": "English"},
    "Spanish": {"qid": "Q1321", "ru": "испанский язык", "en": "Spanish"},
    "French": {"qid": "Q150", "ru": "французский язык", "en": "French"},
    "Arabic": {"qid": "Q13955", "ru": "арабский язык", "en": "Arabic"},
    "Portuguese": {"qid": "Q5146", "ru": "португальский язык", "en": "Portuguese"},
    "Russian": {"qid": "Q7737", "ru": "русский язык", "en": "Russian"},
    "German": {"qid": "Q188", "ru": "немецкий язык", "en": "German"},
    "Dutch": {"qid": "Q7411", "ru": "нидерландский язык", "en": "Dutch"},
    "Swahili": {"qid": "Q7838", "ru": "суахили", "en": "Swahili"},
    "Malay": {"qid": "Q9237", "ru": "малайский язык", "en": "Malay"},
    "Chinese": {"qid": "Q7850", "ru": "китайский язык", "en": "Chinese"},
}

CURRENCIES: Dict[str, Dict[str, str]] = {
    "euro": {"qid": "Q4916", "ru": "евро", "en": "euro"},
    "United States dollar": {"qid": "Q4917", "ru": "доллар США", "en": "United States dollar"},
    "pound sterling": {"qid": "Q25224", "ru": "фунт стерлингов", "en": "pound sterling"},
    "Japanese yen": {"qid": "Q8146", "ru": "японская иена", "en": "Japanese yen"},
    "CFA franc BCEAO": {"qid": "Q861690", "ru": "франк КФА BCEAO", "en": "CFA franc BCEAO"},
    "CFA franc BEAC": {"qid": "Q847739", "ru": "франк КФА BEAC", "en": "CFA franc BEAC"},
    "Australian dollar": {"qid": "Q259502", "ru": "австралийский доллар", "en": "Australian dollar"},
    "New Zealand dollar": {"qid": "Q1472704", "ru": "новозеландский доллар", "en": "New Zealand dollar"},
    "East Caribbean dollar": {"qid": "Q26365", "ru": "восточно-карибский доллар", "en": "East Caribbean dollar"},
}

ORGANIZATIONS: Dict[str, Dict[str, str]] = {
    "European Union": {"qid": "Q458", "ru": "Европейский союз", "en": "European Union"},
    "African Union": {"qid": "Q7159", "ru": "Африканский союз", "en": "African Union"},
    "Commonwealth of Nations": {"qid": "Q7785", "ru": "Содружество наций", "en": "Commonwealth of Nations"},
    "NATO": {"qid": "Q7184", "ru": "НАТО", "en": "NATO"},
    "OECD": {"qid": "Q41550", "ru": "ОЭСР", "en": "OECD"},
    "OPEC": {"qid": "Q7795", "ru": "ОПЕК", "en": "OPEC"},
    "Arab League": {"qid": "Q7172", "ru": "Лига арабских государств", "en": "Arab League"},
    "Mercosur": {"qid": "Q4264", "ru": "Меркосур", "en": "Mercosur"},
}

# English display names for natural text. The values stay human-readable; constraints still use clean names above.
ORGANIZATION_EN_TEXT = {
    "European Union": "the European Union",
    "African Union": "the African Union",
    "Commonwealth of Nations": "the Commonwealth of Nations",
    "Arab League": "the Arab League",
    "OECD": "the OECD",
    "OPEC": "OPEC",
    "NATO": "NATO",
    "Mercosur": "Mercosur",
}

# Known Wikidata P37 false positives that should not become gold answers for natural-language
# questions about a country's official language. These are applied as explicit FILTERs in SPARQL
# and also used by resume-quality checks to drop older rows.
COUNTRIES_FALSE_OFFICIAL_LANGUAGE_QID_EXCLUSIONS = {
    "Arabic": ["Q889"],   # Afghanistan: official languages are Pashto and Dari, not Arabic.
    "Spanish": ["Q30"],  # United States: no federal official language; Spanish is not a sovereign-state official language.
}

# Curated combinations are intentionally broad enough to produce >=5 golds but selective enough for useful benchmark questions.
L1_COMBOS = [
    ("continent_language", "Africa", "French"),
    ("continent_language", "Africa", "Arabic"),
    ("continent_language", "Europe", "German"),
    ("continent_language", "Europe", "English"),
    ("continent_language", "South America", "Spanish"),
    ("continent_language", "North America", "English"),
    ("continent_currency", "Europe", "euro"),
    ("continent_currency", "Africa", "CFA franc BCEAO"),
    ("continent_currency", "Oceania", "Australian dollar"),
    ("org_language", "Commonwealth of Nations", "English"),
    ("org_language", "African Union", "French"),
    ("org_language", "Arab League", "Arabic"),
    ("org_currency", "European Union", "euro"),
    ("org_currency", "OPEC", "United States dollar"),
    ("org_currency", "OECD", "euro"),
]

L2_COMBOS = [
    ("continent_language_currency", "Africa", "French", "CFA franc BCEAO"),
    ("continent_language_currency", "Africa", "French", "CFA franc BEAC"),
    ("continent_language_currency", "Europe", "German", "euro"),
    ("continent_language_currency", "South America", "Spanish", "United States dollar"),
    ("continent_language_currency", "Oceania", "English", "Australian dollar"),
    ("continent_language_org", "Africa", "Arabic", "Arab League"),
    ("continent_language_org", "Africa", "French", "African Union"),
    ("continent_language_org", "Europe", "English", "NATO"),
    ("continent_language_org", "Europe", "French", "European Union"),
    ("continent_population_language", "Asia", "Arabic", 1_000_000, 50_000_000),
    ("continent_population_language", "Africa", "French", 1_000_000, 80_000_000),
    ("continent_population_language", "Europe", "English", 1_000_000, 80_000_000),
    ("org_population_language", "OECD", "English", 1_000_000, 100_000_000),
    ("org_population_language", "Commonwealth of Nations", "English", 1_000_000, 100_000_000),
    ("org_population_language", "African Union", "French", 1_000_000, 100_000_000),
    ("org_currency_language", "European Union", "euro", "German"),
    ("org_currency_language", "African Union", "CFA franc BCEAO", "French"),
    ("org_currency_language", "Commonwealth of Nations", "East Caribbean dollar", "English"),
    # Additional high-yield positive L2 patterns. These avoid unsafe negative membership and
    # give L2 enough valid diversity without waiting through hundreds of impossible combos.
    ("continent_currency_org", "Europe", "euro", "European Union"),
    ("continent_currency_org", "Europe", "euro", "OECD"),
    ("continent_currency_org", "Africa", "CFA franc BCEAO", "African Union"),
    ("continent_currency_org", "Africa", "CFA franc BEAC", "African Union"),
    ("continent_currency_org", "North America", "East Caribbean dollar", "Commonwealth of Nations"),
    ("continent_language_org", "Africa", "English", "Commonwealth of Nations"),
    ("continent_language_org", "Africa", "Portuguese", "African Union"),
    ("continent_language_org", "Oceania", "English", "Commonwealth of Nations"),
    ("continent_population_currency", "Europe", "euro", 1_000_000, 100_000_000),
    ("continent_population_currency", "Africa", "CFA franc BCEAO", 1_000_000, 50_000_000),
    ("continent_population_currency", "Africa", "CFA franc BEAC", 1_000_000, 50_000_000),
    ("continent_population_currency", "North America", "East Caribbean dollar", 40_000, 200_000),
    ("org_population_currency", "European Union", "euro", 1_000_000, 100_000_000),
    ("org_population_currency", "OECD", "euro", 1_000_000, 100_000_000),
    ("org_population_currency", "Commonwealth of Nations", "East Caribbean dollar", 40_000, 200_000),
]

# Template internals below use these metadata maps for readable SPARQL documentation.
COUNTRY_PROPERTY_PATHS = {
    "country_type": "P31 exact sovereign state (Q3624078)",
    "current_country_filter": "P31=Q3624078 plus FILTER NOT EXISTS P576/P582",
    "continent": "P30",
    "official_language": "P37",
    "currency": "P38",
    "population": "P1082",
    "capital": "P36",
    "capital_population": "P36/P1082",
    "bordering_country": "P47",
    "member_of_current": "p:P463/ps:P463 plus no pq:P582",
    "heritage_site_in_country": "inverse P17 + P1435=UNESCO World Heritage Site (Q9259)",
    "large_city_in_country": "inverse P17 + P31/P279*=city + P1082",
}


In [4]:
# ============================================================
# SPARQL building, gold collection, and schema helpers
# ============================================================

PREFIXES = """PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX wikibase: <http://wikiba.se/ontology#>
PREFIX p: <http://www.wikidata.org/prop/>
PREFIX ps: <http://www.wikidata.org/prop/statement/>
PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
""".strip()


def _c_norm(x: Any) -> str:
    if x is None:
        return ""
    s = str(x).strip()
    return "" if s.lower() in {"nan", "none", "null"} else s


def _c_now() -> str:
    return dt.datetime.utcnow().replace(microsecond=0).isoformat() + "Z"


def _c_clean_constraints(d: Dict[str, Any]) -> Dict[str, Any]:
    out: Dict[str, Any] = {}
    for k, v in (d or {}).items():
        if v is None:
            continue
        if isinstance(v, str) and not v.strip():
            continue
        if isinstance(v, (list, tuple)) and len(v) == 0:
            continue
        if isinstance(v, dict):
            vv = _c_clean_constraints(v)
            if vv:
                out[k] = vv
            continue
        out[k] = v
    return out


def _c_json_default(x: Any):
    if isinstance(x, set):
        return sorted(x)
    if isinstance(x, Path):
        return str(x)
    try:
        import numpy as np
        if isinstance(x, np.integer):
            return int(x)
        if isinstance(x, np.floating):
            return float(x)
    except Exception:
        pass
    return str(x)


def _c_country_base_lines(var: str = "?country") -> List[str]:
    # IMPORTANT: exact sovereign-state class, not broad country/subclass traversal.
    # The broad pattern `P31/P279* Q6256` leaks constituent countries, dependent territories,
    # micronations and fictional/self-proclaimed projects into gold lists.
    return [
        f"{var} wdt:P31 wd:{Q_SOVEREIGN_STATE} .",
        f"FILTER NOT EXISTS {{ {var} wdt:P576 ?country_dissolved . }}",
        f"FILTER NOT EXISTS {{ {var} wdt:P582 ?country_end . }}",
    ]


def _c_current_country_lines(var: str, prefix: str) -> List[str]:
    return [
        f"{var} wdt:P31 wd:{Q_SOVEREIGN_STATE} .",
        f"FILTER NOT EXISTS {{ {var} wdt:P576 ?{prefix}_dissolved . }}",
        f"FILTER NOT EXISTS {{ {var} wdt:P582 ?{prefix}_end . }}",
    ]


def _c_current_member_lines(country_var: str, org_qid: str, stmt_prefix: str) -> List[str]:
    stmt = f"?{stmt_prefix}_membershipStatement"
    end = f"?{stmt_prefix}_membershipEnd"
    return [
        f"{country_var} p:P463 {stmt} .",
        f"{stmt} ps:P463 wd:{org_qid} .",
        f"FILTER NOT EXISTS {{ {stmt} pq:P582 {end} . }}",
    ]


def _c_not_current_member_line(country_var: str, org_qid: str, stmt_prefix: str) -> str:
    stmt = f"?{stmt_prefix}_excludedMembershipStatement"
    end = f"?{stmt_prefix}_excludedMembershipEnd"
    return (
        f"FILTER NOT EXISTS {{ {country_var} p:P463 {stmt} . "
        f"{stmt} ps:P463 wd:{org_qid} . "
        f"FILTER NOT EXISTS {{ {stmt} pq:P582 {end} . }} }}"
    )


def _c_build_select(where_lines: Sequence[str], limit: int = COUNTRIES_WDQS_LIMIT) -> str:
    body = "\n      ".join(where_lines)
    return f"""{PREFIXES}
SELECT DISTINCT ?country ?countryLabelEn ?countryLabelRu WHERE {{
      {body}
      ?country rdfs:label ?countryLabelEn FILTER(LANG(?countryLabelEn) = "en") .
      OPTIONAL {{ ?country rdfs:label ?countryLabelRu FILTER(LANG(?countryLabelRu) = "ru") . }}
}}
LIMIT {int(limit)}""".strip()


def _c_build_ask(where_lines: Sequence[str]) -> str:
    body = "\n      ".join(where_lines)
    return f"""{PREFIXES}
# WDQS-only validator. Replace {{ITEM}} with a country QID.
ASK WHERE {{
      BIND(wd:{{ITEM}} AS ?country)
      {body}
}}""".strip()

_COUNTRIES_SPARQL_CACHE: Dict[str, List[Dict[str, Any]]] = {}
_COUNTRIES_FAILED_SPARQL: Dict[str, str] = {}


def _c_select_rows_cached(sparql: str) -> Tuple[List[Dict[str, Any]], Optional[str]]:
    if sparql in _COUNTRIES_SPARQL_CACHE:
        return list(_COUNTRIES_SPARQL_CACHE[sparql]), None
    if sparql in _COUNTRIES_FAILED_SPARQL:
        return [], _COUNTRIES_FAILED_SPARQL[sparql]
    try:
        rows = rows_from_select(wd.sparql_select(sparql))
        _COUNTRIES_SPARQL_CACHE[sparql] = list(rows)
        return list(rows), None
    except Exception as e:
        msg = f"{type(e).__name__}: {e}"
        _COUNTRIES_FAILED_SPARQL[sparql] = msg
        return [], msg


def _c_collect_gold(spec: Dict[str, Any]) -> Tuple[Optional[Dict[str, Any]], Dict[str, Any]]:
    sparql = _c_build_select(spec["where_lines"], limit=spec.get("limit", COUNTRIES_WDQS_LIMIT))
    ask = _c_build_ask(spec["where_lines"])
    rows, err = _c_select_rows_cached(sparql)
    if err:
        return None, {"reason": "wdqs_error", "error": err, "template_id": spec.get("template_id")}

    qids: List[str] = []
    labels_ru: List[str] = []
    labels_en: List[str] = []
    seen: set = set()
    dropped = {"missing_qid": 0, "missing_en_label": 0, "duplicate_qid": 0}
    ru_count = 0
    fallback_ru = 0

    for r in rows:
        qid = uri_to_qid(r.get("country", ""))
        en = _c_norm(r.get("countryLabelEn", ""))
        ru = _c_norm(r.get("countryLabelRu", ""))
        if not qid:
            dropped["missing_qid"] += 1
            continue
        if not en:
            dropped["missing_en_label"] += 1
            continue
        if qid in seen:
            dropped["duplicate_qid"] += 1
            continue
        seen.add(qid)
        qids.append(qid)
        labels_en.append(en)
        if ru:
            labels_ru.append(ru)
            ru_count += 1
        else:
            labels_ru.append(en)
            fallback_ru += 1

    gold_count = len(qids)
    min_gold = int(spec.get("min_gold", COUNTRIES_ACCEPT_MIN_GOLD.get(spec["complexity"], 5)))
    max_gold = int(spec.get("max_gold", COUNTRIES_ACCEPT_MAX_GOLD.get(spec["complexity"], 80)))
    returned_limit = int(spec.get("limit", COUNTRIES_WDQS_LIMIT))
    truncated = len(rows) >= returned_limit
    ru_share = (ru_count / gold_count) if gold_count else 0.0

    if gold_count < min_gold:
        return None, {"reason": "too_few_gold", "gold_count": gold_count, "min_gold": min_gold, "template_id": spec.get("template_id")}
    if gold_count > max_gold:
        return None, {"reason": "too_many_gold", "gold_count": gold_count, "max_gold": max_gold, "template_id": spec.get("template_id")}
    if truncated:
        return None, {"reason": "wdqs_limit_reached", "rows_returned": len(rows), "limit": returned_limit, "template_id": spec.get("template_id")}
    if ru_share < float(COUNTRIES_MIN_RU_LABEL_SHARE):
        return None, {"reason": "low_ru_label_share", "ru_share": ru_share, "gold_count": gold_count, "template_id": spec.get("template_id")}

    record = {
        "id": spec["id"],
        "domain": COUNTRIES_DOMAIN,
        "complexity": spec["complexity"],
        "query_text_ru": spec["query_text_ru"],
        "constraints": _c_clean_constraints(spec.get("constraints", {})),
        "requested_count": int(spec.get("requested_count", 5)),
        "gold_answer_qids": qids[:300],
        "gold_answer_labels_ru": labels_ru[:300],
        "sparql_query": sparql,
        "created_at": _c_now(),
        "query_text_en": spec["query_text_en"],
        "gold_answer_labels_en": labels_en[:300],
        "is_advanced": bool(spec.get("is_advanced", spec["complexity"] in {"L3", "L4", "L5"})),
        "template_id": spec.get("template_id", "countries_unknown"),
        "template_family": spec.get("template_family", "countries"),
        "gold_truncated": False,
        "ask_validator_sparql": ask,
        "local_validator": {
            "type": "none_wdqs_only",
            "source": "Wikidata Query Service",
            "match_key": "wikidata_qid",
            "applies_after": "ask_validator_sparql",
            "filters": _c_clean_constraints(spec.get("constraints", {})),
            "label_matching_used": False,
            "note": "Use ask_validator_sparql for all Wikidata constraints; no external local validator is required for countries-domain examples.",
        },
        "gold_collection_meta": {
            "source": "wikidata_sparql",
            "match_key": "wikidata_qid",
            "candidates_from_wdqs": len(rows),
            "wdqs_candidate_limit": returned_limit,
            "rows_returned_by_wdqs": len(rows),
            "gold_returned_before_limits": gold_count,
            "dropped_reasons": dropped,
            "dropped_no_qid_count": dropped["missing_qid"],
            "dropped_no_en_label_count": dropped["missing_en_label"],
            "label_sources": {"ru_label": ru_count, "en_fallback_for_ru": fallback_ru},
            "note": "Gold is collected directly from Wikidata SPARQL. English label is required; Russian label uses English fallback when absent.",
            "gold_may_be_incomplete_due_to_wdqs_limit": False,
            "constraints_are_wdqs_only": True,
            "gold_limit": 300,
            "gold_returned": gold_count,
            "gold_total_before_limit": gold_count,
            "gold_truncated_by_local_limit": False,
            "bridge_meta": spec.get("bridge_meta", {}),
        },
        "gold_answer_imdb_ids": [],
        "gold_answer_imdb_titles": [],
    }
    return record, {"reason": "accepted", "gold_count": gold_count, "template_id": spec.get("template_id")}


def _c_read_jsonl(path: Path) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    rows: List[Dict[str, Any]] = []
    bad: List[Dict[str, Any]] = []
    if not Path(path).exists():
        return rows, bad
    with open(path, "r", encoding="utf-8") as f:
        for lineno, line in enumerate(f, 1):
            s = line.strip()
            if not s:
                continue
            try:
                rows.append(json.loads(s))
            except Exception as e:
                bad.append({"line": lineno, "error": str(e), "text": s[:300]})
    return rows, bad


def _c_append_jsonl(path: Path, record: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False, default=_c_json_default) + "\n")
        f.flush()


def _c_rewrite_jsonl(path: Path, records: Sequence[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False, default=_c_json_default) + "\n")
    tmp.replace(path)


def _c_signature(record_or_spec: Dict[str, Any]) -> Tuple[Any, ...]:
    constraints = record_or_spec.get("constraints", {}) or {}
    return (
        record_or_spec.get("complexity"),
        record_or_spec.get("template_id"),
        json.dumps(_c_clean_constraints(constraints), ensure_ascii=False, sort_keys=True, default=_c_json_default),
    )


def _c_query_signature(record_or_spec: Dict[str, Any]) -> str:
    return re.sub(r"\s+", " ", _c_norm(record_or_spec.get("query_text_ru", "")).lower())


def _c_gold_count(r: Dict[str, Any]) -> int:
    return len(r.get("gold_answer_qids") or [])


def _c_find_reference_jsonl_paths() -> List[Path]:
    names = ["countries.jsonl", "countries_final_curated.jsonl", "countries_final_curated_v2.jsonl"]
    roots: List[Path] = []
    try:
        cwd = Path.cwd()
        roots.extend([cwd, cwd.parent, cwd / "out_wikidata_benchmark/domain_outputs", cwd.parent / "out_wikidata_benchmark/domain_outputs"])
    except Exception:
        pass
    roots.extend([COUNTRIES_OUTPUT_DIR])
    out: List[Path] = []
    seen: set = set()
    for p in list(COUNTRIES_REFERENCE_JSONL_PATHS):
        try:
            rp = p.resolve()
        except Exception:
            rp = p
        if p.exists() and str(rp) not in seen:
            seen.add(str(rp)); out.append(p)
    for root in roots:
        for name in names:
            p = root / name
            try:
                rp = p.resolve()
            except Exception:
                rp = p
            if p.exists() and str(rp) not in seen:
                seen.add(str(rp)); out.append(p)
    return out


In [5]:
# ============================================================
# Candidate spec builders
# ============================================================

def _ent(table: Dict[str, Dict[str, str]], key: str) -> Dict[str, str]:
    if key not in table:
        raise KeyError(key)
    return table[key]


def _base_spec(complexity: str, template_id: str, template_family: str, constraints: Dict[str, Any], where_lines: List[str], query_ru: str, query_en: str, bridge: str, constraint_entities: Dict[str, str], property_paths: Dict[str, str], is_advanced: Optional[bool] = None) -> Dict[str, Any]:
    clean_constraints = _c_clean_constraints({"kind": "country", "country_status": "current", **constraints})
    # Apply targeted false-positive filters to the target country. This keeps gold answers factual
    # while preserving clean human-readable constraints.
    where_lines = list(where_lines) + _c_false_official_language_filter_lines("?country", clean_constraints.get("official_language"))
    return {
        "complexity": complexity,
        "template_id": template_id,
        "template_family": template_family,
        "constraints": clean_constraints,
        "where_lines": where_lines,
        "query_text_ru": _c_clean_query_text(query_ru),
        "query_text_en": _c_clean_query_text(query_en),
        "requested_count": 5,
        "is_advanced": (complexity in {"L3", "L4", "L5"}) if is_advanced is None else bool(is_advanced),
        "bridge_meta": {
            "bridge": bridge,
            "constraint_entities": constraint_entities,
            "constraint_property_paths": property_paths,
        },
        "limit": COUNTRIES_WDQS_LIMIT,
        "min_gold": COUNTRIES_ACCEPT_MIN_GOLD.get(complexity, 5),
        "max_gold": COUNTRIES_ACCEPT_MAX_GOLD.get(complexity, 80),
    }


def _query_head_ru() -> str:
    return "Назови 5 современных государств"


def _query_head_en() -> str:
    return "Name 5 current sovereign states"


def _c_clean_query_text(s: str) -> str:
    s = re.sub(r"\s+", " ", str(s or "")).strip()
    s = s.replace(" ,", ",")
    return s


def _fmt_num_ru(n: int) -> str:
    return f"{int(n):,}".replace(",", " ")


def _fmt_num_en(n: int) -> str:
    return f"{int(n):,}"


def _org_en_text(org: Dict[str, str]) -> str:
    return ORGANIZATION_EN_TEXT.get(org.get("en", ""), org.get("en", ""))


def _org_ru_text(org: Dict[str, str]) -> str:
    return f"организацию «{org['ru']}»"


def _c_false_official_language_filter_lines(country_var: str, lang_name: Optional[str]) -> List[str]:
    qids = COUNTRIES_FALSE_OFFICIAL_LANGUAGE_QID_EXCLUSIONS.get(str(lang_name or ""), [])
    return [f"FILTER({country_var} != wd:{qid})" for qid in qids]


def spec_l1_continent_language(cont_key: str, lang_key: str) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); lang = _ent(LANGUAGES, lang_key)
    where = _c_country_base_lines("?country") + [f"?country wdt:P30 wd:{cont['qid']} .", f"?country wdt:P37 wd:{lang['qid']} ."]
    return _base_spec(
        "L1", "countries_l1_continent_language", "country_basic_filters",
        {"continent": cont["en"], "official_language": lang["en"]}, where,
        f"{_query_head_ru()}, которые находятся в регионе «{cont['ru']}» и имеют официальный язык «{lang['ru']}».",
        f"{_query_head_en()} located in {cont['en']} whose official language is {lang['en']}.",
        "country -> continent and official language",
        {"continent_qid": cont["qid"], "official_language_qid": lang["qid"]},
        {"continent": "P30", "official_language": "P37", "country_status": COUNTRY_PROPERTY_PATHS["current_country_filter"]},
        is_advanced=False,
    )


def spec_l1_continent_currency(cont_key: str, curr_key: str) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); curr = _ent(CURRENCIES, curr_key)
    where = _c_country_base_lines("?country") + [f"?country wdt:P30 wd:{cont['qid']} .", f"?country wdt:P38 wd:{curr['qid']} ."]
    return _base_spec(
        "L1", "countries_l1_continent_currency", "country_basic_filters",
        {"continent": cont["en"], "currency": curr["en"]}, where,
        f"{_query_head_ru()}, которые находятся в регионе «{cont['ru']}» и используют валюту «{curr['ru']}».",
        f"{_query_head_en()} located in {cont['en']} that use the currency {curr['en']}.",
        "country -> continent and currency",
        {"continent_qid": cont["qid"], "currency_qid": curr["qid"]},
        {"continent": "P30", "currency": "P38", "country_status": COUNTRY_PROPERTY_PATHS["current_country_filter"]},
        is_advanced=False,
    )


def spec_l1_org_language(org_key: str, lang_key: str) -> Dict[str, Any]:
    org = _ent(ORGANIZATIONS, org_key); lang = _ent(LANGUAGES, lang_key)
    where = _c_country_base_lines("?country") + _c_current_member_lines("?country", org["qid"], "org") + [f"?country wdt:P37 wd:{lang['qid']} ."]
    return _base_spec(
        "L1", "countries_l1_current_member_language", "country_basic_filters",
        {"member_of": org["en"], "official_language": lang["en"]}, where,
        f"{_query_head_ru()}, которые сейчас входят в организацию «{org['ru']}» и имеют официальный язык «{lang['ru']}».",
        f"{_query_head_en()} that are current members of {_org_en_text(org)} and whose official language is {lang['en']}.",
        "country -> current membership and official language",
        {"member_of_qid": org["qid"], "official_language_qid": lang["qid"]},
        {"member_of_current": COUNTRY_PROPERTY_PATHS["member_of_current"], "official_language": "P37"},
        is_advanced=False,
    )


def spec_l1_org_currency(org_key: str, curr_key: str) -> Dict[str, Any]:
    org = _ent(ORGANIZATIONS, org_key); curr = _ent(CURRENCIES, curr_key)
    where = _c_country_base_lines("?country") + _c_current_member_lines("?country", org["qid"], "org") + [f"?country wdt:P38 wd:{curr['qid']} ."]
    return _base_spec(
        "L1", "countries_l1_current_member_currency", "country_basic_filters",
        {"member_of": org["en"], "currency": curr["en"]}, where,
        f"{_query_head_ru()}, которые сейчас входят в организацию «{org['ru']}» и используют валюту «{curr['ru']}».",
        f"{_query_head_en()} that are current members of {_org_en_text(org)} and use the currency {curr['en']}.",
        "country -> current membership and currency",
        {"member_of_qid": org["qid"], "currency_qid": curr["qid"]},
        {"member_of_current": COUNTRY_PROPERTY_PATHS["member_of_current"], "currency": "P38"},
        is_advanced=False,
    )


def spec_l2_continent_language_currency(cont_key: str, lang_key: str, curr_key: str) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); lang = _ent(LANGUAGES, lang_key); curr = _ent(CURRENCIES, curr_key)
    where = _c_country_base_lines("?country") + [f"?country wdt:P30 wd:{cont['qid']} .", f"?country wdt:P37 wd:{lang['qid']} .", f"?country wdt:P38 wd:{curr['qid']} ."]
    return _base_spec(
        "L2", "countries_l2_continent_language_currency", "country_compound_filters",
        {"continent": cont["en"], "official_language": lang["en"], "currency": curr["en"]}, where,
        f"{_query_head_ru()} в регионе «{cont['ru']}», где официальный язык — «{lang['ru']}», а валюта — «{curr['ru']}».",
        f"{_query_head_en()} in {cont['en']} whose official language is {lang['en']} and whose currency is {curr['en']}.",
        "country -> continent, official language, currency",
        {"continent_qid": cont["qid"], "official_language_qid": lang["qid"], "currency_qid": curr["qid"]},
        {"continent": "P30", "official_language": "P37", "currency": "P38"},
        is_advanced=False,
    )


def spec_l2_continent_language_org(cont_key: str, lang_key: str, org_key: str) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); lang = _ent(LANGUAGES, lang_key); org = _ent(ORGANIZATIONS, org_key)
    where = _c_country_base_lines("?country") + [f"?country wdt:P30 wd:{cont['qid']} .", f"?country wdt:P37 wd:{lang['qid']} ."] + _c_current_member_lines("?country", org["qid"], "org")
    return _base_spec(
        "L2", "countries_l2_continent_language_member", "country_compound_filters",
        {"continent": cont["en"], "official_language": lang["en"], "member_of": org["en"]}, where,
        f"{_query_head_ru()} в регионе «{cont['ru']}», которые сейчас входят в {_org_ru_text(org)} и имеют официальный язык «{lang['ru']}».",
        f"{_query_head_en()} in {cont['en']} that are current members of {_org_en_text(org)} and whose official language is {lang['en']}.",
        "country -> continent, official language, current membership",
        {"continent_qid": cont["qid"], "official_language_qid": lang["qid"], "member_of_qid": org["qid"]},
        {"continent": "P30", "official_language": "P37", "member_of_current": COUNTRY_PROPERTY_PATHS["member_of_current"]},
        is_advanced=False,
    )


def spec_l2_continent_currency_not_org(cont_key: str, curr_key: str, org_key: str) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); curr = _ent(CURRENCIES, curr_key); org = _ent(ORGANIZATIONS, org_key)
    where = _c_country_base_lines("?country") + [f"?country wdt:P30 wd:{cont['qid']} .", f"?country wdt:P38 wd:{curr['qid']} .", _c_not_current_member_line("?country", org["qid"], "org")]
    return _base_spec(
        "L2", "countries_l2_continent_currency_not_member", "country_compound_filters",
        {"continent": cont["en"], "currency": curr["en"], "not_member_of": org["en"]}, where,
        f"{_query_head_ru()} в регионе «{cont['ru']}», которые используют валюту «{curr['ru']}» и сейчас не входят в «{org['ru']}».",
        f"{_query_head_en()} in {cont['en']} that use the currency {curr['en']} and are not current members of {_org_en_text(org)}.",
        "country -> continent, currency, not current member of organization",
        {"continent_qid": cont["qid"], "currency_qid": curr["qid"], "not_member_of_qid": org["qid"]},
        {"continent": "P30", "currency": "P38", "not_member_of_current": COUNTRY_PROPERTY_PATHS["member_of_current"]},
        is_advanced=False,
    )



def spec_l2_continent_currency_org(cont_key: str, curr_key: str, org_key: str) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); curr = _ent(CURRENCIES, curr_key); org = _ent(ORGANIZATIONS, org_key)
    where = _c_country_base_lines("?country") + [f"?country wdt:P30 wd:{cont['qid']} .", f"?country wdt:P38 wd:{curr['qid']} ."] + _c_current_member_lines("?country", org["qid"], "org")
    return _base_spec(
        "L2", "countries_l2_continent_currency_member", "country_compound_filters",
        {"continent": cont["en"], "currency": curr["en"], "member_of": org["en"]}, where,
        f"{_query_head_ru()} в регионе «{cont['ru']}», которые сейчас входят в {_org_ru_text(org)} и используют валюту «{curr['ru']}».",
        f"{_query_head_en()} in {cont['en']} that are current members of {_org_en_text(org)} and use the currency {curr['en']}.",
        "country -> continent, currency, current membership",
        {"continent_qid": cont["qid"], "currency_qid": curr["qid"], "member_of_qid": org["qid"]},
        {"continent": "P30", "currency": "P38", "member_of_current": COUNTRY_PROPERTY_PATHS["member_of_current"]},
        is_advanced=False,
    )


def spec_l2_population_currency(scope_kind: str, scope_key: str, curr_key: str, pop_min: int, pop_max: int) -> Dict[str, Any]:
    curr = _ent(CURRENCIES, curr_key)
    where = _c_country_base_lines("?country") + [f"?country wdt:P38 wd:{curr['qid']} .", "?country wdt:P1082 ?countryPopulation .", f"FILTER(?countryPopulation >= {int(pop_min)})", f"FILTER(?countryPopulation <= {int(pop_max)})"]
    constraints = {"currency": curr["en"], "population_min": int(pop_min), "population_max": int(pop_max)}
    ent_qids = {"currency_qid": curr["qid"]}
    if scope_kind == "continent":
        cont = _ent(CONTINENTS, scope_key)
        where.append(f"?country wdt:P30 wd:{cont['qid']} .")
        constraints["continent"] = cont["en"]
        ent_qids["continent_qid"] = cont["qid"]
        ru_scope = f"в регионе «{cont['ru']}»"
        en_scope = f"in {cont['en']}"
    else:
        org = _ent(ORGANIZATIONS, scope_key)
        where += _c_current_member_lines("?country", org["qid"], "org")
        constraints["member_of"] = org["en"]
        ent_qids["member_of_qid"] = org["qid"]
        ru_scope = f"которые сейчас входят в {_org_ru_text(org)}"
        en_scope = f"that are current members of {_org_en_text(org)}"
    return _base_spec(
        "L2", f"countries_l2_{scope_kind}_population_currency", "country_compound_filters",
        constraints, where,
        (f"{_query_head_ru()} {ru_scope}, которые используют валюту «{curr['ru']}» и имеют население от {_fmt_num_ru(pop_min)} до {_fmt_num_ru(pop_max)} человек."
         if scope_kind == "continent" else
         f"{_query_head_ru()}, {ru_scope}, используют валюту «{curr['ru']}» и имеют население от {_fmt_num_ru(pop_min)} до {_fmt_num_ru(pop_max)} человек."),
        (f"{_query_head_en()} {en_scope} that use the currency {curr['en']} and have a population between {_fmt_num_en(pop_min)} and {_fmt_num_en(pop_max)}."
         if scope_kind == "continent" else
         f"{_query_head_en()} {en_scope}, use the currency {curr['en']} and have a population between {_fmt_num_en(pop_min)} and {_fmt_num_en(pop_max)}."),
        "country -> scope, currency, population range",
        ent_qids,
        {"currency": "P38", "population": "P1082", "scope": "P30 or current P463"},
        is_advanced=False,
    )


def spec_l2_population_language(scope_kind: str, scope_key: str, lang_key: str, pop_min: int, pop_max: int) -> Dict[str, Any]:
    lang = _ent(LANGUAGES, lang_key)
    where = _c_country_base_lines("?country") + [f"?country wdt:P37 wd:{lang['qid']} .", "?country wdt:P1082 ?countryPopulation .", f"FILTER(?countryPopulation >= {int(pop_min)})", f"FILTER(?countryPopulation <= {int(pop_max)})"]
    constraints = {"official_language": lang["en"], "population_min": int(pop_min), "population_max": int(pop_max)}
    ent_qids = {"official_language_qid": lang["qid"]}
    if scope_kind == "continent":
        cont = _ent(CONTINENTS, scope_key)
        where.append(f"?country wdt:P30 wd:{cont['qid']} .")
        constraints["continent"] = cont["en"]
        ent_qids["continent_qid"] = cont["qid"]
        ru_scope = f"в регионе «{cont['ru']}»"
        en_scope = f"in {cont['en']}"
    else:
        org = _ent(ORGANIZATIONS, scope_key)
        where += _c_current_member_lines("?country", org["qid"], "org")
        constraints["member_of"] = org["en"]
        ent_qids["member_of_qid"] = org["qid"]
        ru_scope = f"которые сейчас входят в {_org_ru_text(org)}"
        en_scope = f"that are current members of {_org_en_text(org)}"
    return _base_spec(
        "L2", f"countries_l2_{scope_kind}_population_language", "country_compound_filters",
        constraints, where,
        (f"{_query_head_ru()} {ru_scope} с официальным языком «{lang['ru']}» и населением от {_fmt_num_ru(pop_min)} до {_fmt_num_ru(pop_max)} человек."
         if scope_kind == "continent" else
         f"{_query_head_ru()}, {ru_scope}, имеют официальный язык «{lang['ru']}» и население от {_fmt_num_ru(pop_min)} до {_fmt_num_ru(pop_max)} человек."),
        (f"{_query_head_en()} {en_scope} whose official language is {lang['en']} and whose population is between {_fmt_num_en(pop_min)} and {_fmt_num_en(pop_max)}."
         if scope_kind == "continent" else
         f"{_query_head_en()} {en_scope}, whose official language is {lang['en']} and whose population is between {_fmt_num_en(pop_min)} and {_fmt_num_en(pop_max)}."),
        "country -> scope, official language, population range",
        ent_qids,
        {"official_language": "P37", "population": "P1082", "scope": "P30 or current P463"},
        is_advanced=False,
    )


def spec_l2_org_currency_language(org_key: str, curr_key: str, lang_key: str) -> Dict[str, Any]:
    org = _ent(ORGANIZATIONS, org_key); curr = _ent(CURRENCIES, curr_key); lang = _ent(LANGUAGES, lang_key)
    where = _c_country_base_lines("?country") + _c_current_member_lines("?country", org["qid"], "org") + [f"?country wdt:P38 wd:{curr['qid']} .", f"?country wdt:P37 wd:{lang['qid']} ."]
    return _base_spec(
        "L2", "countries_l2_member_currency_language", "country_compound_filters",
        {"member_of": org["en"], "currency": curr["en"], "official_language": lang["en"]}, where,
        f"{_query_head_ru()}, которые сейчас входят в {_org_ru_text(org)}, используют валюту «{curr['ru']}» и имеют официальный язык «{lang['ru']}».",
        f"{_query_head_en()} that are current members of {_org_en_text(org)}, use the currency {curr['en']}, and whose official language is {lang['en']}.",
        "country -> current membership, currency, official language",
        {"member_of_qid": org["qid"], "currency_qid": curr["qid"], "official_language_qid": lang["qid"]},
        {"member_of_current": COUNTRY_PROPERTY_PATHS["member_of_current"], "currency": "P38", "official_language": "P37"},
        is_advanced=False,
    )


def spec_l3_capital_population(cont_key: str, lang_key: str, cap_min: int, cap_max: Optional[int] = None) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); lang = _ent(LANGUAGES, lang_key)
    where = _c_country_base_lines("?country") + [f"?country wdt:P30 wd:{cont['qid']} .", f"?country wdt:P37 wd:{lang['qid']} .", "?country wdt:P36 ?capital .", "?capital wdt:P1082 ?capitalPopulation .", f"FILTER(?capitalPopulation >= {int(cap_min)})"]
    constraints = {"continent": cont["en"], "official_language": lang["en"], "capital_population_min": int(cap_min)}
    if cap_max is not None:
        where.append(f"FILTER(?capitalPopulation <= {int(cap_max)})")
        constraints["capital_population_max"] = int(cap_max)
    return _base_spec(
        "L3", "countries_l3_continent_language_capital_population", "capital_bridge",
        constraints, where,
        f"{_query_head_ru()} в регионе «{cont['ru']}», где официальный язык — «{lang['ru']}», а население столицы не меньше {_fmt_num_ru(cap_min)} человек.",
        f"{_query_head_en()} in {cont['en']} whose official language is {lang['en']} and whose capital has a population of at least {_fmt_num_en(cap_min)}.",
        "country -> capital -> capital population, plus continent and official language",
        {"continent_qid": cont["qid"], "official_language_qid": lang["qid"]},
        {"continent": "P30", "official_language": "P37", "capital_population": COUNTRY_PROPERTY_PATHS["capital_population"]},
    )


def spec_l3_has_whs(cont_key: str, lang_key: str) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); lang = _ent(LANGUAGES, lang_key)
    where = _c_country_base_lines("?country") + [f"?country wdt:P30 wd:{cont['qid']} .", f"?country wdt:P37 wd:{lang['qid']} .", f"?site wdt:P17 ?country .", f"?site wdt:P1435 wd:{Q_UNESCO_WHS} ."]
    return _base_spec(
        "L3", "countries_l3_continent_language_unesco_site", "heritage_site_bridge",
        {"continent": cont["en"], "official_language": lang["en"], "has_unesco_world_heritage_site": True}, where,
        f"{_query_head_ru()} в регионе «{cont['ru']}», где официальный язык — «{lang['ru']}», и есть объект Всемирного наследия ЮНЕСКО.",
        f"{_query_head_en()} in {cont['en']} whose official language is {lang['en']} and that have a UNESCO World Heritage Site.",
        "country <- site located in country, site has UNESCO World Heritage Site designation",
        {"continent_qid": cont["qid"], "official_language_qid": lang["qid"], "unesco_world_heritage_site_qid": Q_UNESCO_WHS},
        {"heritage_site_in_country": COUNTRY_PROPERTY_PATHS["heritage_site_in_country"], "continent": "P30", "official_language": "P37"},
    )


def spec_l3_bordering_language_currency(cont_key: str, own_lang_key: str, neighbor_curr_key: str) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); lang = _ent(LANGUAGES, own_lang_key); curr = _ent(CURRENCIES, neighbor_curr_key)
    where = _c_country_base_lines("?country") + [f"?country wdt:P30 wd:{cont['qid']} .", f"?country wdt:P37 wd:{lang['qid']} .", "?country wdt:P47 ?neighbor ."] + _c_current_country_lines("?neighbor", "neighbor") + [f"?neighbor wdt:P38 wd:{curr['qid']} .", "FILTER(?neighbor != ?country)"]
    return _base_spec(
        "L3", "countries_l3_border_neighbor_currency", "border_bridge",
        {"continent": cont["en"], "official_language": lang["en"], "bordering_country_currency": curr["en"]}, where,
        f"{_query_head_ru()} в регионе «{cont['ru']}», где официальный язык — «{lang['ru']}», и которые граничат со страной, использующей валюту «{curr['ru']}».",
        f"{_query_head_en()} in {cont['en']} whose official language is {lang['en']} and that border a country using the currency {curr['en']}.",
        "country -> bordering country -> bordering country currency",
        {"continent_qid": cont["qid"], "official_language_qid": lang["qid"], "bordering_country_currency_qid": curr["qid"]},
        {"bordering_country": "P47", "bordering_country_currency": "P47/P38", "continent": "P30", "official_language": "P37"},
    )


def spec_l3_member_large_city(org_key: str, city_min: int, lang_key: Optional[str] = None) -> Dict[str, Any]:
    org = _ent(ORGANIZATIONS, org_key)
    where = _c_country_base_lines("?country") + _c_current_member_lines("?country", org["qid"], "org") + [f"?city wdt:P31/wdt:P279* wd:{Q_CITY} .", "?city wdt:P17 ?country .", "?city wdt:P1082 ?cityPopulation .", f"FILTER(?cityPopulation >= {int(city_min)})"]
    constraints = {"member_of": org["en"], "has_city_population_min": int(city_min)}
    ent = {"member_of_qid": org["qid"], "city_qid": Q_CITY}
    ru_lang = ""
    en_lang = ""
    if lang_key:
        lang = _ent(LANGUAGES, lang_key)
        where.append(f"?country wdt:P37 wd:{lang['qid']} .")
        constraints["official_language"] = lang["en"]
        ent["official_language_qid"] = lang["qid"]
        ru_lang = f", имеют официальный язык «{lang['ru']}»"
        en_lang = f", whose official language is {lang['en']}"
    return _base_spec(
        "L3", "countries_l3_member_large_city", "large_city_bridge",
        constraints, where,
        f"{_query_head_ru()}, которые сейчас входят в {_org_ru_text(org)}{ru_lang} и имеют город с населением не меньше {_fmt_num_ru(city_min)} человек.",
        f"{_query_head_en()} that are current members of {_org_en_text(org)}{en_lang} and have a city with a population of at least {_fmt_num_en(city_min)}.",
        "country <- city located in country, city population; plus current membership",
        ent,
        {"member_of_current": COUNTRY_PROPERTY_PATHS["member_of_current"], "large_city_in_country": COUNTRY_PROPERTY_PATHS["large_city_in_country"], "official_language": "P37"},
    )


def spec_l4_border_member_not_member(cont_key: str, lang_key: str, border_org_key: str, not_org_key: str) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); lang = _ent(LANGUAGES, lang_key); borg = _ent(ORGANIZATIONS, border_org_key); norg = _ent(ORGANIZATIONS, not_org_key)
    where = _c_country_base_lines("?country") + [f"?country wdt:P30 wd:{cont['qid']} .", f"?country wdt:P37 wd:{lang['qid']} .", _c_not_current_member_line("?country", norg["qid"], "notorg"), "?country wdt:P47 ?neighbor ."] + _c_current_country_lines("?neighbor", "neighbor") + _c_current_member_lines("?neighbor", borg["qid"], "neighborOrg") + ["FILTER(?neighbor != ?country)"]
    return _base_spec(
        "L4", "countries_l4_border_member_not_member", "border_membership_bridge",
        {"continent": cont["en"], "official_language": lang["en"], "bordering_country_member_of": borg["en"], "not_member_of": norg["en"]}, where,
        f"{_query_head_ru()} в регионе «{cont['ru']}», где официальный язык — «{lang['ru']}», которые сейчас не входят в «{norg['ru']}» и граничат со страной-членом «{borg['ru']}».",
        f"{_query_head_en()} in {cont['en']} whose official language is {lang['en']}, that are not current members of {_org_en_text(norg)} and border a current member of {_org_en_text(borg)}.",
        "country -> border country -> border country current membership; country not current member of another organization",
        {"continent_qid": cont["qid"], "official_language_qid": lang["qid"], "bordering_country_member_of_qid": borg["qid"], "not_member_of_qid": norg["qid"]},
        {"bordering_country": "P47", "bordering_country_member_of_current": "P47 + current P463", "not_member_of_current": COUNTRY_PROPERTY_PATHS["member_of_current"], "continent": "P30", "official_language": "P37"},
    )


def spec_l4_member_whs_capital(org_key: str, cap_min: int, curr_key: Optional[str] = None) -> Dict[str, Any]:
    org = _ent(ORGANIZATIONS, org_key)
    where = _c_country_base_lines("?country") + _c_current_member_lines("?country", org["qid"], "org") + [f"?site wdt:P17 ?country .", f"?site wdt:P1435 wd:{Q_UNESCO_WHS} .", "?country wdt:P36 ?capital .", "?capital wdt:P1082 ?capitalPopulation .", f"FILTER(?capitalPopulation >= {int(cap_min)})"]
    constraints = {"member_of": org["en"], "has_unesco_world_heritage_site": True, "capital_population_min": int(cap_min)}
    ent = {"member_of_qid": org["qid"], "unesco_world_heritage_site_qid": Q_UNESCO_WHS}
    ru_curr = ""
    en_curr = ""
    if curr_key:
        curr = _ent(CURRENCIES, curr_key)
        where.append(f"?country wdt:P38 wd:{curr['qid']} .")
        constraints["currency"] = curr["en"]
        ent["currency_qid"] = curr["qid"]
        ru_curr = f", используют валюту «{curr['ru']}»"
        en_curr = f", use the currency {curr['en']}"
    return _base_spec(
        "L4", "countries_l4_member_whs_capital_population", "capital_heritage_membership_bridge",
        constraints, where,
        f"{_query_head_ru()}, которые сейчас входят в {_org_ru_text(org)}{ru_curr}, имеют объект Всемирного наследия ЮНЕСКО и столицу с населением не меньше {_fmt_num_ru(cap_min)} человек.",
        f"{_query_head_en()} that are current members of {_org_en_text(org)}{en_curr}, have a UNESCO World Heritage Site, and have a capital with a population of at least {_fmt_num_en(cap_min)}.",
        "country -> current membership; country <- UNESCO site; country -> capital -> population",
        ent,
        {"member_of_current": COUNTRY_PROPERTY_PATHS["member_of_current"], "heritage_site_in_country": COUNTRY_PROPERTY_PATHS["heritage_site_in_country"], "capital_population": COUNTRY_PROPERTY_PATHS["capital_population"], "currency": "P38"},
    )


def spec_l4_currency_border_org_population(cont_key: str, curr_key: str, border_org_key: str, pop_min: int) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); curr = _ent(CURRENCIES, curr_key); borg = _ent(ORGANIZATIONS, border_org_key)
    where = _c_country_base_lines("?country") + [f"?country wdt:P30 wd:{cont['qid']} .", f"?country wdt:P38 wd:{curr['qid']} .", "?country wdt:P1082 ?countryPopulation .", f"FILTER(?countryPopulation >= {int(pop_min)})", "?country wdt:P47 ?neighbor ."] + _c_current_country_lines("?neighbor", "neighbor") + _c_current_member_lines("?neighbor", borg["qid"], "neighborOrg") + ["FILTER(?neighbor != ?country)"]
    return _base_spec(
        "L4", "countries_l4_currency_border_org_population", "border_membership_bridge",
        {"continent": cont["en"], "currency": curr["en"], "population_min": int(pop_min), "bordering_country_member_of": borg["en"]}, where,
        f"{_query_head_ru()} в регионе «{cont['ru']}», которые используют валюту «{curr['ru']}», имеют население не меньше {_fmt_num_ru(pop_min)} человек и граничат со страной, которая сейчас входит в {_org_ru_text(borg)}.",
        f"{_query_head_en()} in {cont['en']} that use the currency {curr['en']}, have a population of at least {_fmt_num_en(pop_min)}, and border a current member of {_org_en_text(borg)}.",
        "country -> border country -> border country current membership, plus country population and currency",
        {"continent_qid": cont["qid"], "currency_qid": curr["qid"], "bordering_country_member_of_qid": borg["qid"]},
        {"continent": "P30", "currency": "P38", "population": "P1082", "bordering_country_member_of_current": "P47 + current P463"},
    )



# ============================================================
# Additional high-recall safe L4/L5 templates
# These avoid negative membership and use bridges that produce enough countries:
# UNESCO site, capital population, large city, and bordering-country attributes.
# ============================================================

def spec_l4_continent_language_whs_capital(cont_key: str, lang_key: str, cap_min: int) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); lang = _ent(LANGUAGES, lang_key)
    where = (
        _c_country_base_lines("?country")
        + [f"?country wdt:P30 wd:{cont['qid']} .", f"?country wdt:P37 wd:{lang['qid']} .",
           f"?site wdt:P17 ?country .", f"?site wdt:P1435 wd:{Q_UNESCO_WHS} .",
           "?country wdt:P36 ?capital .", "?capital wdt:P1082 ?capitalPopulation .",
           f"FILTER(?capitalPopulation >= {int(cap_min)})"]
    )
    return _base_spec(
        "L4", "countries_l4_continent_language_whs_capital_population", "capital_heritage_bridge",
        {"continent": cont["en"], "official_language": lang["en"], "has_unesco_world_heritage_site": True, "capital_population_min": int(cap_min)}, where,
        f"{_query_head_ru()} в регионе «{cont['ru']}», где официальный язык — «{lang['ru']}», у которых есть объект Всемирного наследия ЮНЕСКО и столица с населением не меньше {_fmt_num_ru(cap_min)} человек.",
        f"{_query_head_en()} in {cont['en']} whose official language is {lang['en']}, and that have a UNESCO World Heritage Site and a capital with a population of at least {_fmt_num_en(cap_min)}.",
        "country -> capital -> population; country <- UNESCO site; plus continent and official language",
        {"continent_qid": cont["qid"], "official_language_qid": lang["qid"], "unesco_world_heritage_site_qid": Q_UNESCO_WHS},
        {"continent": "P30", "official_language": "P37", "heritage_site_in_country": COUNTRY_PROPERTY_PATHS["heritage_site_in_country"], "capital_population": COUNTRY_PROPERTY_PATHS["capital_population"]},
    )


def spec_l4_continent_currency_whs_capital(cont_key: str, curr_key: str, cap_min: int) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); curr = _ent(CURRENCIES, curr_key)
    where = (
        _c_country_base_lines("?country")
        + [f"?country wdt:P30 wd:{cont['qid']} .", f"?country wdt:P38 wd:{curr['qid']} .",
           f"?site wdt:P17 ?country .", f"?site wdt:P1435 wd:{Q_UNESCO_WHS} .",
           "?country wdt:P36 ?capital .", "?capital wdt:P1082 ?capitalPopulation .",
           f"FILTER(?capitalPopulation >= {int(cap_min)})"]
    )
    return _base_spec(
        "L4", "countries_l4_continent_currency_whs_capital_population", "capital_heritage_bridge",
        {"continent": cont["en"], "currency": curr["en"], "has_unesco_world_heritage_site": True, "capital_population_min": int(cap_min)}, where,
        f"{_query_head_ru()} в регионе «{cont['ru']}», которые используют валюту «{curr['ru']}», имеют объект Всемирного наследия ЮНЕСКО и столицу с населением не меньше {_fmt_num_ru(cap_min)} человек.",
        f"{_query_head_en()} in {cont['en']} that use the currency {curr['en']}, have a UNESCO World Heritage Site, and have a capital with a population of at least {_fmt_num_en(cap_min)}.",
        "country -> capital -> population; country <- UNESCO site; plus continent and currency",
        {"continent_qid": cont["qid"], "currency_qid": curr["qid"], "unesco_world_heritage_site_qid": Q_UNESCO_WHS},
        {"continent": "P30", "currency": "P38", "heritage_site_in_country": COUNTRY_PROPERTY_PATHS["heritage_site_in_country"], "capital_population": COUNTRY_PROPERTY_PATHS["capital_population"]},
    )


def spec_l4_org_language_whs_capital(org_key: str, lang_key: str, cap_min: int) -> Dict[str, Any]:
    org = _ent(ORGANIZATIONS, org_key); lang = _ent(LANGUAGES, lang_key)
    where = (
        _c_country_base_lines("?country")
        + _c_current_member_lines("?country", org["qid"], "org")
        + [f"?country wdt:P37 wd:{lang['qid']} .",
           f"?site wdt:P17 ?country .", f"?site wdt:P1435 wd:{Q_UNESCO_WHS} .",
           "?country wdt:P36 ?capital .", "?capital wdt:P1082 ?capitalPopulation .",
           f"FILTER(?capitalPopulation >= {int(cap_min)})"]
    )
    return _base_spec(
        "L4", "countries_l4_org_language_whs_capital_population", "capital_heritage_membership_bridge",
        {"member_of": org["en"], "official_language": lang["en"], "has_unesco_world_heritage_site": True, "capital_population_min": int(cap_min)}, where,
        f"{_query_head_ru()}, которые сейчас входят в {_org_ru_text(org)}, имеют официальный язык «{lang['ru']}», объект Всемирного наследия ЮНЕСКО и столицу с населением не меньше {_fmt_num_ru(cap_min)} человек.",
        f"{_query_head_en()} that are current members of {_org_en_text(org)}, whose official language is {lang['en']}, and that have a UNESCO World Heritage Site and a capital with a population of at least {_fmt_num_en(cap_min)}.",
        "country -> current membership; country -> capital -> population; country <- UNESCO site; plus official language",
        {"member_of_qid": org["qid"], "official_language_qid": lang["qid"], "unesco_world_heritage_site_qid": Q_UNESCO_WHS},
        {"member_of_current": COUNTRY_PROPERTY_PATHS["member_of_current"], "official_language": "P37", "heritage_site_in_country": COUNTRY_PROPERTY_PATHS["heritage_site_in_country"], "capital_population": COUNTRY_PROPERTY_PATHS["capital_population"]},
    )


def spec_l4_continent_language_large_city_whs(cont_key: str, lang_key: str, city_min: int) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); lang = _ent(LANGUAGES, lang_key)
    where = (
        _c_country_base_lines("?country")
        + [f"?country wdt:P30 wd:{cont['qid']} .", f"?country wdt:P37 wd:{lang['qid']} .",
           f"?site wdt:P17 ?country .", f"?site wdt:P1435 wd:{Q_UNESCO_WHS} .",
           f"?city wdt:P31/wdt:P279* wd:{Q_CITY} .", "?city wdt:P17 ?country .", "?city wdt:P1082 ?cityPopulation .",
           f"FILTER(?cityPopulation >= {int(city_min)})"]
    )
    return _base_spec(
        "L4", "countries_l4_continent_language_whs_large_city", "large_city_heritage_bridge",
        {"continent": cont["en"], "official_language": lang["en"], "has_unesco_world_heritage_site": True, "has_city_population_min": int(city_min)}, where,
        f"{_query_head_ru()} в регионе «{cont['ru']}», где официальный язык — «{lang['ru']}», у которых есть объект Всемирного наследия ЮНЕСКО и город с населением не меньше {_fmt_num_ru(city_min)} человек.",
        f"{_query_head_en()} in {cont['en']} whose official language is {lang['en']}, and that have a UNESCO World Heritage Site and a city with a population of at least {_fmt_num_en(city_min)}.",
        "country <- UNESCO site; country <- large city; plus continent and official language",
        {"continent_qid": cont["qid"], "official_language_qid": lang["qid"], "unesco_world_heritage_site_qid": Q_UNESCO_WHS},
        {"continent": "P30", "official_language": "P37", "heritage_site_in_country": COUNTRY_PROPERTY_PATHS["heritage_site_in_country"], "large_city_in_country": COUNTRY_PROPERTY_PATHS["large_city_in_country"]},
    )


def spec_l5_continent_language_whs_capital_border_org(cont_key: str, lang_key: str, cap_min: int, border_org_key: str) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); lang = _ent(LANGUAGES, lang_key); borg = _ent(ORGANIZATIONS, border_org_key)
    where = (
        _c_country_base_lines("?country")
        + [f"?country wdt:P30 wd:{cont['qid']} .", f"?country wdt:P37 wd:{lang['qid']} .",
           f"?site wdt:P17 ?country .", f"?site wdt:P1435 wd:{Q_UNESCO_WHS} .",
           "?country wdt:P36 ?capital .", "?capital wdt:P1082 ?capitalPopulation .",
           f"FILTER(?capitalPopulation >= {int(cap_min)})", "?country wdt:P47 ?neighbor ."]
        + _c_current_country_lines("?neighbor", "neighbor")
        + _c_current_member_lines("?neighbor", borg["qid"], "neighborOrg")
        + ["FILTER(?neighbor != ?country)"]
    )
    return _base_spec(
        "L5", "countries_l5_continent_language_whs_capital_border_org", "hard_multibridge_country",
        {"continent": cont["en"], "official_language": lang["en"], "has_unesco_world_heritage_site": True, "capital_population_min": int(cap_min), "bordering_country_member_of": borg["en"]}, where,
        f"{_query_head_ru()} в регионе «{cont['ru']}», где официальный язык — «{lang['ru']}», у которых есть объект Всемирного наследия ЮНЕСКО, столица с населением не меньше {_fmt_num_ru(cap_min)} человек и граница со страной, которая сейчас входит в {_org_ru_text(borg)}.",
        f"{_query_head_en()} in {cont['en']} whose official language is {lang['en']}, and that have a UNESCO World Heritage Site, have a capital with a population of at least {_fmt_num_en(cap_min)}, and border a current member of {_org_en_text(borg)}.",
        "country -> capital -> population; country <- UNESCO site; country -> border country -> current membership; plus continent/language",
        {"continent_qid": cont["qid"], "official_language_qid": lang["qid"], "bordering_country_member_of_qid": borg["qid"], "unesco_world_heritage_site_qid": Q_UNESCO_WHS},
        {"continent": "P30", "official_language": "P37", "heritage_site_in_country": COUNTRY_PROPERTY_PATHS["heritage_site_in_country"], "capital_population": COUNTRY_PROPERTY_PATHS["capital_population"], "bordering_country_member_of_current": "P47 + current P463"},
    )


def spec_l5_continent_currency_whs_capital_border_org(cont_key: str, curr_key: str, cap_min: int, border_org_key: str) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); curr = _ent(CURRENCIES, curr_key); borg = _ent(ORGANIZATIONS, border_org_key)
    where = (
        _c_country_base_lines("?country")
        + [f"?country wdt:P30 wd:{cont['qid']} .", f"?country wdt:P38 wd:{curr['qid']} .",
           f"?site wdt:P17 ?country .", f"?site wdt:P1435 wd:{Q_UNESCO_WHS} .",
           "?country wdt:P36 ?capital .", "?capital wdt:P1082 ?capitalPopulation .",
           f"FILTER(?capitalPopulation >= {int(cap_min)})", "?country wdt:P47 ?neighbor ."]
        + _c_current_country_lines("?neighbor", "neighbor")
        + _c_current_member_lines("?neighbor", borg["qid"], "neighborOrg")
        + ["FILTER(?neighbor != ?country)"]
    )
    return _base_spec(
        "L5", "countries_l5_continent_currency_whs_capital_border_org", "hard_multibridge_country",
        {"continent": cont["en"], "currency": curr["en"], "has_unesco_world_heritage_site": True, "capital_population_min": int(cap_min), "bordering_country_member_of": borg["en"]}, where,
        f"{_query_head_ru()} в регионе «{cont['ru']}», которые используют валюту «{curr['ru']}», имеют объект Всемирного наследия ЮНЕСКО, столицу с населением не меньше {_fmt_num_ru(cap_min)} человек и граничат со страной, которая сейчас входит в {_org_ru_text(borg)}.",
        f"{_query_head_en()} in {cont['en']} that use the currency {curr['en']}, have a UNESCO World Heritage Site, have a capital with a population of at least {_fmt_num_en(cap_min)}, and border a current member of {_org_en_text(borg)}.",
        "country -> capital -> population; country <- UNESCO site; country -> border country -> current membership; plus continent/currency",
        {"continent_qid": cont["qid"], "currency_qid": curr["qid"], "bordering_country_member_of_qid": borg["qid"], "unesco_world_heritage_site_qid": Q_UNESCO_WHS},
        {"continent": "P30", "currency": "P38", "heritage_site_in_country": COUNTRY_PROPERTY_PATHS["heritage_site_in_country"], "capital_population": COUNTRY_PROPERTY_PATHS["capital_population"], "bordering_country_member_of_current": "P47 + current P463"},
    )


def spec_l5_org_language_whs_capital_large_city(org_key: str, lang_key: str, cap_min: int, city_min: int) -> Dict[str, Any]:
    org = _ent(ORGANIZATIONS, org_key); lang = _ent(LANGUAGES, lang_key)
    where = (
        _c_country_base_lines("?country")
        + _c_current_member_lines("?country", org["qid"], "org")
        + [f"?country wdt:P37 wd:{lang['qid']} .",
           f"?site wdt:P17 ?country .", f"?site wdt:P1435 wd:{Q_UNESCO_WHS} .",
           "?country wdt:P36 ?capital .", "?capital wdt:P1082 ?capitalPopulation .",
           f"FILTER(?capitalPopulation >= {int(cap_min)})",
           f"?city wdt:P31/wdt:P279* wd:{Q_CITY} .", "?city wdt:P17 ?country .", "?city wdt:P1082 ?cityPopulation .",
           f"FILTER(?cityPopulation >= {int(city_min)})"]
    )
    return _base_spec(
        "L5", "countries_l5_org_language_whs_capital_large_city", "hard_multibridge_country",
        {"member_of": org["en"], "official_language": lang["en"], "has_unesco_world_heritage_site": True, "capital_population_min": int(cap_min), "has_city_population_min": int(city_min)}, where,
        f"{_query_head_ru()}, которые сейчас входят в {_org_ru_text(org)}, имеют официальный язык «{lang['ru']}», объект Всемирного наследия ЮНЕСКО, столицу с населением не меньше {_fmt_num_ru(cap_min)} человек и город с населением не меньше {_fmt_num_ru(city_min)} человек.",
        f"{_query_head_en()} that are current members of {_org_en_text(org)}, whose official language is {lang['en']}, and that have a UNESCO World Heritage Site, a capital with a population of at least {_fmt_num_en(cap_min)}, and a city with a population of at least {_fmt_num_en(city_min)}.",
        "country -> current membership; country <- UNESCO site; country -> capital -> population; country <- large city; plus official language",
        {"member_of_qid": org["qid"], "official_language_qid": lang["qid"], "unesco_world_heritage_site_qid": Q_UNESCO_WHS},
        {"member_of_current": COUNTRY_PROPERTY_PATHS["member_of_current"], "official_language": "P37", "heritage_site_in_country": COUNTRY_PROPERTY_PATHS["heritage_site_in_country"], "capital_population": COUNTRY_PROPERTY_PATHS["capital_population"], "large_city_in_country": COUNTRY_PROPERTY_PATHS["large_city_in_country"]},
    )


def spec_l5_continent_language_whs_large_city_border_neighbor_language(cont_key: str, lang_key: str, city_min: int, neighbor_lang_key: str) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); lang = _ent(LANGUAGES, lang_key); nlang = _ent(LANGUAGES, neighbor_lang_key)
    where = (
        _c_country_base_lines("?country")
        + [f"?country wdt:P30 wd:{cont['qid']} .", f"?country wdt:P37 wd:{lang['qid']} .",
           f"?site wdt:P17 ?country .", f"?site wdt:P1435 wd:{Q_UNESCO_WHS} .",
           f"?city wdt:P31/wdt:P279* wd:{Q_CITY} .", "?city wdt:P17 ?country .", "?city wdt:P1082 ?cityPopulation .",
           f"FILTER(?cityPopulation >= {int(city_min)})", "?country wdt:P47 ?neighbor ."]
        + _c_current_country_lines("?neighbor", "neighbor")
        + [f"?neighbor wdt:P37 wd:{nlang['qid']} .", "FILTER(?neighbor != ?country)"]
        + _c_false_official_language_filter_lines("?neighbor", nlang["en"])
    )
    return _base_spec(
        "L5", "countries_l5_continent_language_whs_large_city_border_neighbor_language", "hard_multibridge_country",
        {"continent": cont["en"], "official_language": lang["en"], "has_unesco_world_heritage_site": True, "has_city_population_min": int(city_min), "bordering_country_official_language": nlang["en"]}, where,
        f"{_query_head_ru()} в регионе «{cont['ru']}», где официальный язык — «{lang['ru']}», у которых есть объект Всемирного наследия ЮНЕСКО, город с населением не меньше {_fmt_num_ru(city_min)} человек и граница со страной, где официальный язык — «{nlang['ru']}».",
        f"{_query_head_en()} in {cont['en']} whose official language is {lang['en']}, and that have a UNESCO World Heritage Site, have a city with a population of at least {_fmt_num_en(city_min)}, and border a country whose official language is {nlang['en']}.",
        "country <- UNESCO site; country <- large city; country -> border country -> official language; plus continent/language",
        {"continent_qid": cont["qid"], "official_language_qid": lang["qid"], "bordering_country_official_language_qid": nlang["qid"], "unesco_world_heritage_site_qid": Q_UNESCO_WHS},
        {"continent": "P30", "official_language": "P37", "heritage_site_in_country": COUNTRY_PROPERTY_PATHS["heritage_site_in_country"], "large_city_in_country": COUNTRY_PROPERTY_PATHS["large_city_in_country"], "bordering_country_language": "P47/P37"},
    )


def spec_l5_full_combo(cont_key: str, lang_key: str, not_org_key: str, border_org_key: str, cap_min: int) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); lang = _ent(LANGUAGES, lang_key); norg = _ent(ORGANIZATIONS, not_org_key); borg = _ent(ORGANIZATIONS, border_org_key)
    where = _c_country_base_lines("?country") + [f"?country wdt:P30 wd:{cont['qid']} .", f"?country wdt:P37 wd:{lang['qid']} .", _c_not_current_member_line("?country", norg["qid"], "notorg"), "?country wdt:P36 ?capital .", "?capital wdt:P1082 ?capitalPopulation .", f"FILTER(?capitalPopulation >= {int(cap_min)})", f"?site wdt:P17 ?country .", f"?site wdt:P1435 wd:{Q_UNESCO_WHS} .", "?country wdt:P47 ?neighbor ."] + _c_current_country_lines("?neighbor", "neighbor") + _c_current_member_lines("?neighbor", borg["qid"], "neighborOrg") + ["FILTER(?neighbor != ?country)"]
    return _base_spec(
        "L5", "countries_l5_continent_language_capital_whs_border_org_not_org", "hard_multibridge_country",
        {"continent": cont["en"], "official_language": lang["en"], "not_member_of": norg["en"], "capital_population_min": int(cap_min), "has_unesco_world_heritage_site": True, "bordering_country_member_of": borg["en"]}, where,
        f"{_query_head_ru()} в регионе «{cont['ru']}», где официальный язык — «{lang['ru']}», которые сейчас не входят в {_org_ru_text(norg)}, имеют объект Всемирного наследия ЮНЕСКО, столицу с населением не меньше {_fmt_num_ru(cap_min)} человек и граничат со страной, которая сейчас входит в {_org_ru_text(borg)}.",
        f"{_query_head_en()} in {cont['en']} whose official language is {lang['en']}, that are not current members of {_org_en_text(norg)}, have a UNESCO World Heritage Site, have a capital with a population of at least {_fmt_num_en(cap_min)}, and border a current member of {_org_en_text(borg)}.",
        "country -> capital -> population; country <- UNESCO site; country -> border country -> membership; country not member of another organization",
        {"continent_qid": cont["qid"], "official_language_qid": lang["qid"], "not_member_of_qid": norg["qid"], "bordering_country_member_of_qid": borg["qid"], "unesco_world_heritage_site_qid": Q_UNESCO_WHS},
        {"continent": "P30", "official_language": "P37", "not_member_of_current": COUNTRY_PROPERTY_PATHS["member_of_current"], "capital_population": COUNTRY_PROPERTY_PATHS["capital_population"], "heritage_site_in_country": COUNTRY_PROPERTY_PATHS["heritage_site_in_country"], "bordering_country_member_of_current": "P47 + current P463"},
    )


def spec_l5_member_border_neighbor_currency_whs(org_key: str, neighbor_lang_key: str, neighbor_curr_key: str, own_curr_key: Optional[str] = None) -> Dict[str, Any]:
    org = _ent(ORGANIZATIONS, org_key); nlang = _ent(LANGUAGES, neighbor_lang_key); ncurr = _ent(CURRENCIES, neighbor_curr_key)
    where = _c_country_base_lines("?country") + _c_current_member_lines("?country", org["qid"], "org") + [f"?site wdt:P17 ?country .", f"?site wdt:P1435 wd:{Q_UNESCO_WHS} .", "?country wdt:P47 ?neighbor ."] + _c_current_country_lines("?neighbor", "neighbor") + [f"?neighbor wdt:P37 wd:{nlang['qid']} .", f"?neighbor wdt:P38 wd:{ncurr['qid']} .", "FILTER(?neighbor != ?country)"]
    constraints = {"member_of": org["en"], "has_unesco_world_heritage_site": True, "bordering_country_official_language": nlang["en"], "bordering_country_currency": ncurr["en"]}
    ent = {"member_of_qid": org["qid"], "bordering_country_official_language_qid": nlang["qid"], "bordering_country_currency_qid": ncurr["qid"], "unesco_world_heritage_site_qid": Q_UNESCO_WHS}
    ru_own = ""
    en_own = ""
    if own_curr_key:
        oc = _ent(CURRENCIES, own_curr_key)
        where.append(f"?country wdt:P38 wd:{oc['qid']} .")
        constraints["currency"] = oc["en"]
        ent["currency_qid"] = oc["qid"]
        ru_own = f", используют валюту «{oc['ru']}»"
        en_own = f", use the currency {oc['en']}"
    return _base_spec(
        "L5", "countries_l5_member_whs_border_neighbor_language_currency", "hard_multibridge_country",
        constraints, where,
        f"{_query_head_ru()}, которые сейчас входят в {_org_ru_text(org)}{ru_own}, имеют объект Всемирного наследия ЮНЕСКО и граничат со страной, где официальный язык — «{nlang['ru']}», а валюта — «{ncurr['ru']}».",
        f"{_query_head_en()} that are current members of {_org_en_text(org)}{en_own}, have a UNESCO World Heritage Site, and border a country whose official language is {nlang['en']} and whose currency is {ncurr['en']}.",
        "country -> membership; country <- UNESCO site; country -> bordering country -> language/currency",
        ent,
        {"member_of_current": COUNTRY_PROPERTY_PATHS["member_of_current"], "heritage_site_in_country": COUNTRY_PROPERTY_PATHS["heritage_site_in_country"], "bordering_country_language": "P47/P37", "bordering_country_currency": "P47/P38", "currency": "P38"},
    )


def spec_l5_shared_currency_border_capital(cont_key: str, lang_key: str, curr_key: str, cap_min: int, city_min: Optional[int] = None) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); lang = _ent(LANGUAGES, lang_key); curr = _ent(CURRENCIES, curr_key)
    where = _c_country_base_lines("?country") + [f"?country wdt:P30 wd:{cont['qid']} .", f"?country wdt:P37 wd:{lang['qid']} .", f"?country wdt:P38 wd:{curr['qid']} .", "?country wdt:P36 ?capital .", "?capital wdt:P1082 ?capitalPopulation .", f"FILTER(?capitalPopulation >= {int(cap_min)})", "?country wdt:P47 ?neighbor ."] + _c_current_country_lines("?neighbor", "neighbor") + [f"?neighbor wdt:P38 wd:{curr['qid']} .", "FILTER(?neighbor != ?country)"]
    constraints = {"continent": cont["en"], "official_language": lang["en"], "currency": curr["en"], "capital_population_min": int(cap_min), "borders_country_with_same_currency": True}
    ent = {"continent_qid": cont["qid"], "official_language_qid": lang["qid"], "currency_qid": curr["qid"]}
    ru_city = ""
    en_city = ""
    if city_min:
        where += [f"?city wdt:P31/wdt:P279* wd:{Q_CITY} .", "?city wdt:P17 ?country .", "?city wdt:P1082 ?cityPopulation .", f"FILTER(?cityPopulation >= {int(city_min)})"]
        constraints["has_city_population_min"] = int(city_min)
        ru_city = f", имеют город с населением не меньше {_fmt_num_ru(city_min)} человек"
        en_city = f", have a city with a population of at least {_fmt_num_en(city_min)}"
    return _base_spec(
        "L5", "countries_l5_shared_currency_border_capital_city", "hard_multibridge_country",
        constraints, where,
        f"{_query_head_ru()} в регионе «{cont['ru']}», где официальный язык — «{lang['ru']}», которые используют валюту «{curr['ru']}», граничат со страной с той же валютой, имеют столицу с населением не меньше {_fmt_num_ru(cap_min)} человек{ru_city}.",
        f"{_query_head_en()} in {cont['en']} whose official language is {lang['en']}, that use the currency {curr['en']}, border a country with the same currency, have a capital with a population of at least {_fmt_num_en(cap_min)}{en_city}.",
        "country -> border country sharing currency; country -> capital -> population; optionally country <- large city",
        ent,
        {"continent": "P30", "official_language": "P37", "currency": "P38", "border_same_currency": "P47 plus both P38", "capital_population": COUNTRY_PROPERTY_PATHS["capital_population"], "large_city_in_country": COUNTRY_PROPERTY_PATHS["large_city_in_country"]},
    )


In [6]:
# ============================================================
# Build deterministic candidate queues
# ============================================================


def _countries_spec_veto_reason(spec: Dict[str, Any]) -> Optional[str]:
    c = spec.get("constraints", {}) or {}
    if "not_member_of" in c:
        return "unsafe_negative_membership"
    # Avoid known unstable generic Asia+Arabic P37 questions unless Arab League membership
    # makes the intended country set unambiguous.
    if c.get("continent") == "Asia" and c.get("official_language") == "Arabic" and c.get("member_of") != "Arab League":
        return "unstable_generic_asia_arabic_official_language"
    return None

def build_countries_candidate_specs(seed: int = COUNTRIES_RANDOM_SEED) -> Dict[str, List[Dict[str, Any]]]:
    rng = random.Random(seed)
    by_level: Dict[str, List[Dict[str, Any]]] = {k: [] for k in COUNTRIES_TARGET_PER_LEVEL}

    for combo in L1_COMBOS:
        kind = combo[0]
        if kind == "continent_language":
            by_level["L1"].append(spec_l1_continent_language(combo[1], combo[2]))
        elif kind == "continent_currency":
            by_level["L1"].append(spec_l1_continent_currency(combo[1], combo[2]))
        elif kind == "org_language":
            by_level["L1"].append(spec_l1_org_language(combo[1], combo[2]))
        elif kind == "org_currency":
            by_level["L1"].append(spec_l1_org_currency(combo[1], combo[2]))

    for combo in L2_COMBOS:
        kind = combo[0]
        if kind == "continent_language_currency":
            by_level["L2"].append(spec_l2_continent_language_currency(combo[1], combo[2], combo[3]))
        elif kind == "continent_language_org":
            by_level["L2"].append(spec_l2_continent_language_org(combo[1], combo[2], combo[3]))
        elif kind == "continent_currency_not_org":
            # Negative membership is unsafe in Wikidata because missing statements can look like non-membership.
            continue
        elif kind == "continent_population_language":
            by_level["L2"].append(spec_l2_population_language("continent", combo[1], combo[2], combo[3], combo[4]))
        elif kind == "org_population_language":
            by_level["L2"].append(spec_l2_population_language("org", combo[1], combo[2], combo[3], combo[4]))
        elif kind == "org_currency_language":
            by_level["L2"].append(spec_l2_org_currency_language(combo[1], combo[2], combo[3]))
        elif kind == "continent_currency_org":
            by_level["L2"].append(spec_l2_continent_currency_org(combo[1], combo[2], combo[3]))
        elif kind == "continent_population_currency":
            by_level["L2"].append(spec_l2_population_currency("continent", combo[1], combo[2], combo[3], combo[4]))
        elif kind == "org_population_currency":
            by_level["L2"].append(spec_l2_population_currency("org", combo[1], combo[2], combo[3], combo[4]))

    # L3: bridges via capital population, UNESCO sites, bordering-country currency, large-city membership.
    for cont, lang, cap in [
        ("Africa", "French", 500_000), ("Africa", "Arabic", 500_000), ("Europe", "English", 500_000),
        ("Europe", "German", 300_000), ("Asia", "Arabic", 1_000_000), ("North America", "English", 500_000),
        ("South America", "Spanish", 500_000), ("Oceania", "English", 100_000),
    ]:
        by_level["L3"].append(spec_l3_capital_population(cont, lang, cap))
    for cont, lang in [
        ("Africa", "French"), ("Africa", "Arabic"), ("Europe", "English"), ("Europe", "German"),
        ("South America", "Spanish"), ("North America", "English"), ("Asia", "Arabic"), ("Oceania", "English"),
    ]:
        by_level["L3"].append(spec_l3_has_whs(cont, lang))
    for cont, lang, curr in [
        ("Europe", "English", "euro"), ("Europe", "German", "euro"), ("Africa", "French", "CFA franc BCEAO"),
        ("Africa", "Arabic", "United States dollar"), ("South America", "Spanish", "United States dollar"),
        ("North America", "English", "United States dollar"),
    ]:
        by_level["L3"].append(spec_l3_bordering_language_currency(cont, lang, curr))
    for org, city_min, lang in [
        ("African Union", 1_000_000, "French"), ("Commonwealth of Nations", 1_000_000, "English"),
        ("European Union", 1_000_000, None), ("OECD", 1_000_000, "English"),
        ("Arab League", 1_000_000, "Arabic"), ("NATO", 1_000_000, None),
    ]:
        by_level["L3"].append(spec_l3_member_large_city(org, city_min, lang))

    # L4: stronger two-bridge patterns.
    # Removed negative-membership L4 templates. They were interesting but unsafe: on Wikidata,
    # absence of a current P463 statement is not a reliable proof of non-membership.
    for args in [
        ("European Union", 500_000, "euro"), ("European Union", 1_000_000, None),
        ("African Union", 500_000, None), ("Commonwealth of Nations", 500_000, "East Caribbean dollar"),
        ("NATO", 500_000, None), ("OECD", 1_000_000, None), ("Arab League", 1_000_000, None),
    ]:
        by_level["L4"].append(spec_l4_member_whs_capital(*args))
    for args in [
        ("Europe", "euro", "NATO", 1_000_000),
        ("Africa", "CFA franc BCEAO", "African Union", 1_000_000),
        ("Africa", "CFA franc BEAC", "African Union", 500_000),
        ("South America", "United States dollar", "Mercosur", 500_000),
        ("Oceania", "Australian dollar", "Commonwealth of Nations", 100_000),
        ("North America", "East Caribbean dollar", "Commonwealth of Nations", 50_000),
    ]:
        by_level["L4"].append(spec_l4_currency_border_org_population(*args))

    # L5: hardest multi-bridge patterns. Deliberately many candidates because some will be too selective.
    # Removed negative-membership L5 templates for the same reason.
    for args in [
        ("European Union", "German", "euro", "euro"),
        ("European Union", "French", "euro", "euro"),
        ("African Union", "French", "CFA franc BCEAO", None),
        ("African Union", "Arabic", "United States dollar", None),
        ("Commonwealth of Nations", "English", "East Caribbean dollar", None),
        ("NATO", "English", "euro", None),
        ("OECD", "English", "United States dollar", None),
        ("Arab League", "Arabic", "United States dollar", None),
    ]:
        by_level["L5"].append(spec_l5_member_border_neighbor_currency_whs(*args))
    for args in [
        ("Europe", "German", "euro", 500_000, None),
        ("Europe", "English", "euro", 500_000, None),
        ("Africa", "French", "CFA franc BCEAO", 300_000, 500_000),
        ("Africa", "French", "CFA franc BEAC", 300_000, None),
        ("North America", "English", "East Caribbean dollar", 50_000, None),
        ("Oceania", "English", "Australian dollar", 100_000, None),
        ("South America", "Spanish", "United States dollar", 500_000, None),
    ]:
        by_level["L5"].append(spec_l5_shared_currency_border_capital(*args))

    # v43 high-recall safe L4 candidates: combine two bridges without unsafe negative membership.
    # These intentionally come before broad backups, so WDQS spends time on candidates that
    # usually have >=5 clean sovereign-state gold answers.
    for args in [
        ("Africa", "French", 300_000), ("Africa", "French", 500_000),
        ("Africa", "Arabic", 300_000), ("Africa", "Arabic", 500_000),
        ("Africa", "English", 300_000), ("Africa", "English", 500_000),
        ("South America", "Spanish", 300_000), ("South America", "Spanish", 500_000),
        ("Europe", "German", 250_000), ("Europe", "German", 500_000),
        ("Europe", "English", 250_000), ("North America", "English", 250_000),
        ("Oceania", "English", 50_000), ("Oceania", "English", 100_000),
    ]:
        by_level["L4"].append(spec_l4_continent_language_whs_capital(*args))
    for args in [
        ("Europe", "euro", 250_000), ("Europe", "euro", 500_000),
        ("Africa", "CFA franc BCEAO", 100_000), ("Africa", "CFA franc BEAC", 100_000),
        ("North America", "East Caribbean dollar", 20_000), ("Oceania", "Australian dollar", 50_000),
    ]:
        by_level["L4"].append(spec_l4_continent_currency_whs_capital(*args))
    for args in [
        ("Commonwealth of Nations", "English", 250_000), ("Commonwealth of Nations", "English", 500_000),
        ("Commonwealth of Nations", "French", 100_000),
        ("African Union", "French", 250_000), ("African Union", "English", 250_000), ("African Union", "Arabic", 250_000),
        ("Arab League", "Arabic", 300_000), ("European Union", "German", 250_000),
        ("European Union", "French", 250_000), ("NATO", "English", 250_000),
    ]:
        by_level["L4"].append(spec_l4_org_language_whs_capital(*args))
    for args in [
        ("Africa", "French", 500_000), ("Africa", "Arabic", 500_000), ("Africa", "English", 500_000),
        ("South America", "Spanish", 500_000), ("Europe", "German", 500_000),
        ("North America", "English", 250_000), ("Oceania", "English", 100_000),
    ]:
        by_level["L4"].append(spec_l4_continent_language_large_city_whs(*args))

    # v43 high-recall safe L5 candidates: same idea, but with three independent bridges.
    for args in [
        ("Africa", "French", 300_000, "African Union"), ("Africa", "French", 500_000, "Commonwealth of Nations"),
        ("Africa", "Arabic", 300_000, "Arab League"), ("Africa", "Arabic", 300_000, "African Union"),
        ("Africa", "English", 300_000, "Commonwealth of Nations"),
        ("South America", "Spanish", 300_000, "Mercosur"),
        ("Europe", "German", 250_000, "European Union"), ("Europe", "German", 250_000, "NATO"),
        ("Europe", "English", 250_000, "NATO"),
        ("North America", "English", 100_000, "Commonwealth of Nations"),
        ("Oceania", "English", 50_000, "Commonwealth of Nations"),
    ]:
        by_level["L5"].append(spec_l5_continent_language_whs_capital_border_org(*args))
    for args in [
        ("Europe", "euro", 250_000, "European Union"), ("Europe", "euro", 250_000, "NATO"),
        ("Africa", "CFA franc BCEAO", 100_000, "African Union"), ("Africa", "CFA franc BCEAO", 100_000, "Commonwealth of Nations"),
        ("Africa", "CFA franc BEAC", 100_000, "African Union"),
        ("North America", "East Caribbean dollar", 20_000, "Commonwealth of Nations"),
    ]:
        by_level["L5"].append(spec_l5_continent_currency_whs_capital_border_org(*args))
    for args in [
        ("Commonwealth of Nations", "English", 250_000, 500_000),
        ("Commonwealth of Nations", "English", 100_000, 250_000),
        ("African Union", "French", 250_000, 500_000),
        ("African Union", "English", 250_000, 500_000),
        ("African Union", "Arabic", 250_000, 500_000),
        ("Arab League", "Arabic", 300_000, 500_000),
        ("European Union", "German", 250_000, 500_000),
        ("European Union", "French", 250_000, 500_000),
    ]:
        by_level["L5"].append(spec_l5_org_language_whs_capital_large_city(*args))
    for args in [
        ("Africa", "French", 500_000, "English"), ("Africa", "French", 500_000, "Arabic"),
        ("Africa", "Arabic", 500_000, "French"), ("Africa", "English", 500_000, "French"),
        ("South America", "Spanish", 500_000, "Spanish"),
        ("Europe", "German", 500_000, "German"), ("Europe", "English", 250_000, "English"),
        ("North America", "English", 250_000, "Spanish"),
    ]:
        by_level["L5"].append(spec_l5_continent_language_whs_large_city_border_neighbor_language(*args))



    # Expand candidate variety with high-recall systematic variants. Curated specs above stay first;
    # extras are used only when a curated candidate is too small/large or duplicated.
    common_langs = ["English", "French", "Spanish", "Arabic", "Portuguese", "German"]
    common_currs = ["euro", "United States dollar", "CFA franc BCEAO", "CFA franc BEAC", "East Caribbean dollar", "Australian dollar"]
    common_orgs = ["African Union", "Commonwealth of Nations", "European Union", "NATO", "OECD", "Arab League", "OPEC", "Mercosur"]

    # L1 backups: lots of simple but valid country filters.
    for cont in ["Africa", "Europe", "Asia", "South America", "North America", "Oceania"]:
        for lang in common_langs:
            if cont in CONTINENTS and lang in LANGUAGES:
                by_level["L1"].append(spec_l1_continent_language(cont, lang))
        for curr in common_currs:
            if cont in CONTINENTS and curr in CURRENCIES:
                by_level["L1"].append(spec_l1_continent_currency(cont, curr))
    for org in common_orgs:
        for lang in common_langs:
            if org in ORGANIZATIONS and lang in LANGUAGES:
                by_level["L1"].append(spec_l1_org_language(org, lang))
        for curr in common_currs:
            if org in ORGANIZATIONS and curr in CURRENCIES:
                by_level["L1"].append(spec_l1_org_currency(org, curr))

    # L2 backups: compound filters.
    for cont in ["Africa", "Europe", "Asia", "South America", "North America", "Oceania"]:
        for lang in common_langs:
            for curr in common_currs:
                if cont in CONTINENTS and lang in LANGUAGES and curr in CURRENCIES:
                    by_level["L2"].append(spec_l2_continent_language_currency(cont, lang, curr))
        for lang in common_langs:
            for org in common_orgs:
                if cont in CONTINENTS and lang in LANGUAGES and org in ORGANIZATIONS:
                    by_level["L2"].append(spec_l2_continent_language_org(cont, lang, org))
        # No not_member_of backups: negative membership is too noisy for country golds.

    # More L2 backups: positive membership/currency/population variants.
    # These are cheaper and higher-recall than generic impossible continent-language-currency grids.
    for cont in ["Africa", "Europe", "Asia", "South America", "North America", "Oceania"]:
        for curr in common_currs:
            for org in common_orgs:
                if cont in CONTINENTS and curr in CURRENCIES and org in ORGANIZATIONS:
                    by_level["L2"].append(spec_l2_continent_currency_org(cont, curr, org))
        for curr in common_currs:
            if cont in CONTINENTS and curr in CURRENCIES:
                by_level["L2"].append(spec_l2_population_currency("continent", cont, curr, 500_000, 20_000_000))
                by_level["L2"].append(spec_l2_population_currency("continent", cont, curr, 1_000_000, 100_000_000))
    for org in common_orgs:
        for curr in common_currs:
            if org in ORGANIZATIONS and curr in CURRENCIES:
                by_level["L2"].append(spec_l2_population_currency("org", org, curr, 500_000, 20_000_000))
                by_level["L2"].append(spec_l2_population_currency("org", org, curr, 1_000_000, 100_000_000))
                for lang in common_langs:
                    if lang in LANGUAGES:
                        by_level["L2"].append(spec_l2_org_currency_language(org, curr, lang))

    # L3 backups.
    for cont in ["Africa", "Europe", "Asia", "South America", "North America", "Oceania"]:
        for lang in common_langs:
            if cont in CONTINENTS and lang in LANGUAGES:
                by_level["L3"].append(spec_l3_has_whs(cont, lang))
                by_level["L3"].append(spec_l3_capital_population(cont, lang, 250_000))
                by_level["L3"].append(spec_l3_capital_population(cont, lang, 1_000_000))
        for lang in common_langs:
            for curr in common_currs:
                if cont in CONTINENTS and lang in LANGUAGES and curr in CURRENCIES:
                    by_level["L3"].append(spec_l3_bordering_language_currency(cont, lang, curr))
    for org in common_orgs:
        if org in ORGANIZATIONS:
            by_level["L3"].append(spec_l3_member_large_city(org, 500_000, None))
            by_level["L3"].append(spec_l3_member_large_city(org, 1_000_000, None))
            for lang in common_langs:
                if lang in LANGUAGES:
                    by_level["L3"].append(spec_l3_member_large_city(org, 500_000, lang))

    # L4 backups.
    for cont in ["Africa", "Europe", "Asia", "South America", "North America", "Oceania"]:
        for curr in common_currs:
            for borg in common_orgs:
                if cont in CONTINENTS and curr in CURRENCIES and borg in ORGANIZATIONS:
                    by_level["L4"].append(spec_l4_currency_border_org_population(cont, curr, borg, 250_000))
                    by_level["L4"].append(spec_l4_currency_border_org_population(cont, curr, borg, 1_000_000))
    for org in common_orgs:
        if org in ORGANIZATIONS:
            by_level["L4"].append(spec_l4_member_whs_capital(org, 250_000, None))
            by_level["L4"].append(spec_l4_member_whs_capital(org, 1_000_000, None))
            for curr in common_currs:
                if curr in CURRENCIES:
                    by_level["L4"].append(spec_l4_member_whs_capital(org, 250_000, curr))

    # L5 backups. These are hard and many will be too selective, so provide a broad queue.
    for cont in ["Africa", "Europe", "Asia", "South America", "North America", "Oceania"]:
        for lang in common_langs:
            for curr in common_currs:
                if cont in CONTINENTS and lang in LANGUAGES and curr in CURRENCIES:
                    by_level["L5"].append(spec_l5_shared_currency_border_capital(cont, lang, curr, 100_000, None))
                    by_level["L5"].append(spec_l5_shared_currency_border_capital(cont, lang, curr, 500_000, None))
    for org in common_orgs:
        for nlang in common_langs:
            for ncurr in common_currs:
                if org in ORGANIZATIONS and nlang in LANGUAGES and ncurr in CURRENCIES:
                    by_level["L5"].append(spec_l5_member_border_neighbor_currency_whs(org, nlang, ncurr, None))


    # De-duplicate specs by clean constraints/template. Keep curated order first for predictable quality.
    for lvl, specs in by_level.items():
        seen = set()
        uniq = []
        for s in specs:
            if _countries_spec_veto_reason(s):
                continue
            sig = _c_signature(s)
            if sig in seen:
                continue
            seen.add(sig)
            uniq.append(s)
        by_level[lvl] = uniq[:COUNTRIES_MAX_CANDIDATES_PER_LEVEL]
    return by_level


COUNTRIES_CANDIDATE_QUEUES = build_countries_candidate_specs(COUNTRIES_RANDOM_SEED)
for lvl, q in COUNTRIES_CANDIDATE_QUEUES.items():
    print(lvl, "candidates:", len(q))


L1 candidates: 167
L2 candidates: 1244
L3 candidates: 390
L4 candidates: 685
L5 candidates: 746


In [7]:

# ============================================================
# v44 rich high-recall templates: events, Olympics, wars, Nobel, capitals on rivers
# ============================================================
# This cell intentionally overrides/extends v43 candidate queues before generation runs.
# Use it for a fresh overgeneration run, then curate/merge with previous countries.jsonl.

COUNTRIES_TARGET_PER_LEVEL = {
    # Generate with запас. Final curated file can be reduced to 100-120 best rows.
    "L1": 18,
    "L2": 22,
    "L3": 35,
    "L4": 55,
    "L5": 55,
}
COUNTRIES_MAX_CANDIDATES_PER_LEVEL = 9000
COUNTRIES_ACCEPT_MIN_GOLD = {"L1": 8, "L2": 6, "L3": 5, "L4": 5, "L5": 5}
COUNTRIES_ACCEPT_MAX_GOLD = {"L1": 95, "L2": 85, "L3": 80, "L4": 85, "L5": 85}

# Allow more hard examples from a useful template family, but still cap primary skew.
COUNTRIES_MAX_SAME_TEMPLATE_PER_LEVEL = {"L1": 7, "L2": 10, "L3": 14, "L4": 34, "L5": 38}
COUNTRIES_MAX_SAME_PRIMARY_CONSTRAINT_PER_LEVEL = {"L1": 6, "L2": 8, "L3": 12, "L4": 16, "L5": 16}

# Countries is small; hard templates often naturally overlap. Keep only near-identical gold duplicates out.
COUNTRIES_DEDUP_GOLD_JACCARD_THRESHOLD = 0.985
COUNTRIES_DEDUP_GOLD_CONTAINMENT_THRESHOLD = 0.999
COUNTRIES_DEDUP_GOLD_SIZE_RATIO_THRESHOLD = 0.98

# Give L4/L5 enough runway before adaptive stop.
COUNTRIES_STAGNATION_LIMIT_BY_LEVEL = {"L1": 150, "L2": 380, "L3": 500, "L4": 1200, "L5": 1600}
COUNTRIES_MIN_ACCEPTED_BEFORE_STAGNATION = {"L1": 12, "L2": 16, "L3": 25, "L4": 42, "L5": 38}

# Extra Wikidata entities for richer positive bridges.
Q_RIVER = "Q4022"
Q_NOBEL_PHYSICS = "Q38104"
Q_NOBEL_CHEMISTRY = "Q44585"
Q_NOBEL_MEDICINE = "Q80061"
Q_NOBEL_LITERATURE = "Q37922"
Q_NOBEL_PEACE = "Q35637"
Q_NOBEL_ECONOMICS = "Q47170"
NOBEL_PRIZE_QIDS = [Q_NOBEL_PHYSICS, Q_NOBEL_CHEMISTRY, Q_NOBEL_MEDICINE, Q_NOBEL_LITERATURE, Q_NOBEL_PEACE, Q_NOBEL_ECONOMICS]

# These QIDs are used as award items on many athlete records.
Q_OLYMPIC_GOLD_MEDAL = "Q15243387"
Q_OLYMPIC_SILVER_MEDAL = "Q15889641"
Q_OLYMPIC_BRONZE_MEDAL = "Q15889643"
OLYMPIC_MEDAL_QIDS = [Q_OLYMPIC_GOLD_MEDAL, Q_OLYMPIC_SILVER_MEDAL, Q_OLYMPIC_BRONZE_MEDAL]

CONFLICTS = {
    "World War I": {"qid": "Q361", "ru": "Первая мировая война", "en": "World War I"},
    "World War II": {"qid": "Q362", "ru": "Вторая мировая война", "en": "World War II"},
    "Korean War": {"qid": "Q8663", "ru": "Корейская война", "en": "Korean War"},
    "Gulf War": {"qid": "Q37643", "ru": "война в Персидском заливе", "en": "Gulf War"},
}

HOSTED_EVENT_TYPES = {
    "Summer Olympic Games": {"qid": "Q159821", "ru": "летние Олимпийские игры", "en": "Summer Olympic Games"},
    "Winter Olympic Games": {"qid": "Q82414", "ru": "зимние Олимпийские игры", "en": "Winter Olympic Games"},
    "FIFA World Cup": {"qid": "Q19317", "ru": "чемпионат мира по футболу FIFA", "en": "FIFA World Cup"},
}

# Extra constraints must remain clean; QIDs stay in bridge_meta only.
def _values_line(var: str, qids: Sequence[str]) -> str:
    return "VALUES " + var + " { " + " ".join(f"wd:{qid}" for qid in qids) + " }"


def _nobel_lines(country_var: str = "?country") -> List[str]:
    return [
        f"?nobelPerson wdt:P27 {country_var} .",
        "?nobelPerson wdt:P166 ?nobelAward .",
        _values_line("?nobelAward", NOBEL_PRIZE_QIDS),
    ]


def _olympic_medalist_lines(country_var: str = "?country") -> List[str]:
    return [
        f"?olympicPerson wdt:P27 {country_var} .",
        "?olympicPerson wdt:P166 ?olympicMedal .",
        _values_line("?olympicMedal", OLYMPIC_MEDAL_QIDS),
    ]


def _hosted_event_lines(country_var: str, event_qid: str, event_var: str = "?hostedEvent") -> List[str]:
    # P31/P279* lets WDQS catch specific event editions typed as subclasses/instances.
    # P17 country is used because it is the most consistently available country-level host relation.
    return [
        f"{event_var} wdt:P31/wdt:P279* wd:{event_qid} .",
        f"{event_var} wdt:P17 {country_var} .",
    ]


def _conflict_participation_lines(country_var: str, conflict_qid: str) -> List[str]:
    return [f"{country_var} wdt:P1344 wd:{conflict_qid} ."]


def _capital_on_river_lines(country_var: str = "?country") -> List[str]:
    return [
        f"{country_var} wdt:P36 ?capital .",
        "?capital wdt:P206 ?capitalWater .",
        f"?capitalWater wdt:P31/wdt:P279* wd:{Q_RIVER} .",
    ]


def _highest_point_lines(country_var: str, min_elevation_m: int) -> List[str]:
    return [
        f"{country_var} wdt:P610 ?highestPoint .",
        "?highestPoint wdt:P2044 ?highestElevation .",
        f"FILTER(?highestElevation >= {int(min_elevation_m)})",
    ]


def spec_l3_continent_language_nobel(cont_key: str, lang_key: str) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); lang = _ent(LANGUAGES, lang_key)
    where = _c_country_base_lines("?country") + [f"?country wdt:P30 wd:{cont['qid']} .", f"?country wdt:P37 wd:{lang['qid']} ."] + _nobel_lines("?country")
    return _base_spec(
        "L3", "countries_l3_continent_language_nobel_citizen", "people_country_bridge",
        {"continent": cont["en"], "official_language": lang["en"], "has_nobel_prize_laureate_citizen": True}, where,
        f"{_query_head_ru()} в регионе «{cont['ru']}», где официальный язык — «{lang['ru']}», и есть гражданин — лауреат Нобелевской премии.",
        f"{_query_head_en()} in {cont['en']} whose official language is {lang['en']} and that have a citizen who is a Nobel Prize laureate.",
        "country <- citizen -> Nobel Prize award, plus continent and official language",
        {"continent_qid": cont["qid"], "official_language_qid": lang["qid"], "nobel_prize_award_qids": NOBEL_PRIZE_QIDS},
        {"continent": "P30", "official_language": "P37", "citizen_nobel_award": "inverse P27 + P166 in Nobel Prize awards"},
    )


def spec_l3_continent_language_olympic_medalist(cont_key: str, lang_key: str) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); lang = _ent(LANGUAGES, lang_key)
    where = _c_country_base_lines("?country") + [f"?country wdt:P30 wd:{cont['qid']} .", f"?country wdt:P37 wd:{lang['qid']} ."] + _olympic_medalist_lines("?country")
    return _base_spec(
        "L3", "countries_l3_continent_language_olympic_medalist_citizen", "people_country_bridge",
        {"continent": cont["en"], "official_language": lang["en"], "has_olympic_medalist_citizen": True}, where,
        f"{_query_head_ru()} в регионе «{cont['ru']}», где официальный язык — «{lang['ru']}», и есть гражданин — призёр Олимпийских игр.",
        f"{_query_head_en()} in {cont['en']} whose official language is {lang['en']} and that have a citizen who is an Olympic medalist.",
        "country <- citizen -> Olympic medal award, plus continent and official language",
        {"continent_qid": cont["qid"], "official_language_qid": lang["qid"], "olympic_medal_award_qids": OLYMPIC_MEDAL_QIDS},
        {"continent": "P30", "official_language": "P37", "citizen_olympic_medal": "inverse P27 + P166 in Olympic medal awards"},
    )


def spec_l3_continent_hosted_event(cont_key: str, event_key: str) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); ev = HOSTED_EVENT_TYPES[event_key]
    where = _c_country_base_lines("?country") + [f"?country wdt:P30 wd:{cont['qid']} ."] + _hosted_event_lines("?country", ev["qid"])
    return _base_spec(
        "L3", "countries_l3_continent_hosted_global_event", "hosted_event_bridge",
        {"continent": cont["en"], "hosted_event_type": ev["en"]}, where,
        f"{_query_head_ru()} в регионе «{cont['ru']}», которые принимали событие типа «{ev['ru']}».",
        f"{_query_head_en()} in {cont['en']} that have hosted an event of type {ev['en']}.",
        "country <- hosted event located in country, plus continent",
        {"continent_qid": cont["qid"], "hosted_event_type_qid": ev["qid"]},
        {"continent": "P30", "hosted_event_type": "inverse P17 + P31/P279*"},
    )


def spec_l3_continent_conflict(cont_key: str, conflict_key: str) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); war = CONFLICTS[conflict_key]
    where = _c_country_base_lines("?country") + [f"?country wdt:P30 wd:{cont['qid']} ."] + _conflict_participation_lines("?country", war["qid"])
    return _base_spec(
        "L3", "countries_l3_continent_participated_in_conflict", "event_participation_bridge",
        {"continent": cont["en"], "participated_in_conflict": war["en"]}, where,
        f"{_query_head_ru()} в регионе «{cont['ru']}», которые участвовали в событии «{war['ru']}».",
        f"{_query_head_en()} in {cont['en']} that participated in {war['en']}.",
        "country -> participant in conflict, plus continent",
        {"continent_qid": cont["qid"], "conflict_qid": war["qid"]},
        {"continent": "P30", "participated_in_conflict": "P1344"},
    )


def spec_l4_language_whs_nobel_capital(cont_key: str, lang_key: str, cap_min: int) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); lang = _ent(LANGUAGES, lang_key)
    where = (
        _c_country_base_lines("?country")
        + [f"?country wdt:P30 wd:{cont['qid']} .", f"?country wdt:P37 wd:{lang['qid']} .",
           f"?site wdt:P17 ?country .", f"?site wdt:P1435 wd:{Q_UNESCO_WHS} .",
           "?country wdt:P36 ?capital .", "?capital wdt:P1082 ?capitalPopulation .", f"FILTER(?capitalPopulation >= {int(cap_min)})"]
        + _nobel_lines("?country")
    )
    return _base_spec(
        "L4", "countries_l4_language_whs_nobel_capital", "people_capital_heritage_bridge",
        {"continent": cont["en"], "official_language": lang["en"], "has_unesco_world_heritage_site": True, "has_nobel_prize_laureate_citizen": True, "capital_population_min": int(cap_min)}, where,
        f"{_query_head_ru()} в регионе «{cont['ru']}», где официальный язык — «{lang['ru']}», есть объект Всемирного наследия ЮНЕСКО, есть гражданин — лауреат Нобелевской премии, а население столицы не меньше {_fmt_num_ru(cap_min)} человек.",
        f"{_query_head_en()} in {cont['en']} whose official language is {lang['en']}, that have a UNESCO World Heritage Site, have a citizen who is a Nobel Prize laureate, and whose capital has a population of at least {_fmt_num_en(cap_min)}.",
        "country <- UNESCO site; country <- Nobel laureate citizen; country -> capital -> population; plus continent/language",
        {"continent_qid": cont["qid"], "official_language_qid": lang["qid"], "unesco_world_heritage_site_qid": Q_UNESCO_WHS, "nobel_prize_award_qids": NOBEL_PRIZE_QIDS},
        {"continent": "P30", "official_language": "P37", "heritage_site_in_country": COUNTRY_PROPERTY_PATHS["heritage_site_in_country"], "citizen_nobel_award": "inverse P27 + P166", "capital_population": COUNTRY_PROPERTY_PATHS["capital_population"]},
    )


def spec_l4_currency_whs_olympic_city(cont_key: str, curr_key: str, city_min: int) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); curr = _ent(CURRENCIES, curr_key)
    where = (
        _c_country_base_lines("?country")
        + [f"?country wdt:P30 wd:{cont['qid']} .", f"?country wdt:P38 wd:{curr['qid']} .",
           f"?site wdt:P17 ?country .", f"?site wdt:P1435 wd:{Q_UNESCO_WHS} .",
           f"?city wdt:P31/wdt:P279* wd:{Q_CITY} .", "?city wdt:P17 ?country .", "?city wdt:P1082 ?cityPopulation .", f"FILTER(?cityPopulation >= {int(city_min)})"]
        + _olympic_medalist_lines("?country")
    )
    return _base_spec(
        "L4", "countries_l4_currency_whs_olympic_city", "sports_heritage_city_bridge",
        {"continent": cont["en"], "currency": curr["en"], "has_unesco_world_heritage_site": True, "has_olympic_medalist_citizen": True, "has_city_population_min": int(city_min)}, where,
        f"{_query_head_ru()} в регионе «{cont['ru']}», которые используют валюту «{curr['ru']}», имеют объект Всемирного наследия ЮНЕСКО, гражданина — призёра Олимпийских игр и город с населением не меньше {_fmt_num_ru(city_min)} человек.",
        f"{_query_head_en()} in {cont['en']} that use the currency {curr['en']}, have a UNESCO World Heritage Site, have a citizen who is an Olympic medalist, and have a city with a population of at least {_fmt_num_en(city_min)}.",
        "country <- UNESCO site; country <- Olympic medalist citizen; country <- large city; plus continent/currency",
        {"continent_qid": cont["qid"], "currency_qid": curr["qid"], "unesco_world_heritage_site_qid": Q_UNESCO_WHS, "olympic_medal_award_qids": OLYMPIC_MEDAL_QIDS},
        {"continent": "P30", "currency": "P38", "heritage_site_in_country": COUNTRY_PROPERTY_PATHS["heritage_site_in_country"], "citizen_olympic_medal": "inverse P27 + P166", "large_city_in_country": COUNTRY_PROPERTY_PATHS["large_city_in_country"]},
    )


def spec_l4_conflict_whs_capital(cont_key: str, conflict_key: str, cap_min: int) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); war = CONFLICTS[conflict_key]
    where = (
        _c_country_base_lines("?country")
        + [f"?country wdt:P30 wd:{cont['qid']} .",
           f"?site wdt:P17 ?country .", f"?site wdt:P1435 wd:{Q_UNESCO_WHS} .",
           "?country wdt:P36 ?capital .", "?capital wdt:P1082 ?capitalPopulation .", f"FILTER(?capitalPopulation >= {int(cap_min)})"]
        + _conflict_participation_lines("?country", war["qid"])
    )
    return _base_spec(
        "L4", "countries_l4_conflict_whs_capital", "event_capital_heritage_bridge",
        {"continent": cont["en"], "participated_in_conflict": war["en"], "has_unesco_world_heritage_site": True, "capital_population_min": int(cap_min)}, where,
        f"{_query_head_ru()} в регионе «{cont['ru']}», которые участвовали в событии «{war['ru']}», имеют объект Всемирного наследия ЮНЕСКО и столицу с населением не меньше {_fmt_num_ru(cap_min)} человек.",
        f"{_query_head_en()} in {cont['en']} that participated in {war['en']}, have a UNESCO World Heritage Site, and have a capital with a population of at least {_fmt_num_en(cap_min)}.",
        "country -> conflict participation; country <- UNESCO site; country -> capital -> population; plus continent",
        {"continent_qid": cont["qid"], "conflict_qid": war["qid"], "unesco_world_heritage_site_qid": Q_UNESCO_WHS},
        {"continent": "P30", "participated_in_conflict": "P1344", "heritage_site_in_country": COUNTRY_PROPERTY_PATHS["heritage_site_in_country"], "capital_population": COUNTRY_PROPERTY_PATHS["capital_population"]},
    )


def spec_l4_capital_river_whs_language(cont_key: str, lang_key: str) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); lang = _ent(LANGUAGES, lang_key)
    where = (_c_country_base_lines("?country") + [f"?country wdt:P30 wd:{cont['qid']} .", f"?country wdt:P37 wd:{lang['qid']} .", f"?site wdt:P17 ?country .", f"?site wdt:P1435 wd:{Q_UNESCO_WHS} ."] + _capital_on_river_lines("?country"))
    return _base_spec(
        "L4", "countries_l4_capital_river_whs_language", "capital_geography_heritage_bridge",
        {"continent": cont["en"], "official_language": lang["en"], "has_unesco_world_heritage_site": True, "capital_located_on_river": True}, where,
        f"{_query_head_ru()} в регионе «{cont['ru']}», где официальный язык — «{lang['ru']}», есть объект Всемирного наследия ЮНЕСКО, а столица расположена на реке или у реки.",
        f"{_query_head_en()} in {cont['en']} whose official language is {lang['en']}, that have a UNESCO World Heritage Site, and whose capital is located on or next to a river.",
        "country -> capital -> adjacent body of water river; country <- UNESCO site; plus continent/language",
        {"continent_qid": cont["qid"], "official_language_qid": lang["qid"], "river_qid": Q_RIVER, "unesco_world_heritage_site_qid": Q_UNESCO_WHS},
        {"continent": "P30", "official_language": "P37", "heritage_site_in_country": COUNTRY_PROPERTY_PATHS["heritage_site_in_country"], "capital_on_river": "P36/P206 + river class"},
    )


def spec_l4_hosted_event_language_capital(cont_key: str, event_key: str, lang_key: str, cap_min: int) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); ev = HOSTED_EVENT_TYPES[event_key]; lang = _ent(LANGUAGES, lang_key)
    where = (_c_country_base_lines("?country") + [f"?country wdt:P30 wd:{cont['qid']} .", f"?country wdt:P37 wd:{lang['qid']} .", "?country wdt:P36 ?capital .", "?capital wdt:P1082 ?capitalPopulation .", f"FILTER(?capitalPopulation >= {int(cap_min)})"] + _hosted_event_lines("?country", ev["qid"]))
    return _base_spec(
        "L4", "countries_l4_hosted_event_language_capital", "hosted_event_capital_bridge",
        {"continent": cont["en"], "official_language": lang["en"], "hosted_event_type": ev["en"], "capital_population_min": int(cap_min)}, where,
        f"{_query_head_ru()} в регионе «{cont['ru']}», где официальный язык — «{lang['ru']}», которые принимали событие типа «{ev['ru']}» и имеют столицу с населением не меньше {_fmt_num_ru(cap_min)} человек.",
        f"{_query_head_en()} in {cont['en']} whose official language is {lang['en']}, that have hosted an event of type {ev['en']} and have a capital with a population of at least {_fmt_num_en(cap_min)}.",
        "country <- hosted event; country -> capital -> population; plus continent/language",
        {"continent_qid": cont["qid"], "official_language_qid": lang["qid"], "hosted_event_type_qid": ev["qid"]},
        {"continent": "P30", "official_language": "P37", "hosted_event_type": "inverse P17 + P31/P279*", "capital_population": COUNTRY_PROPERTY_PATHS["capital_population"]},
    )


def spec_l5_nobel_whs_capital_border_language(cont_key: str, lang_key: str, cap_min: int, neighbor_lang_key: str) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); lang = _ent(LANGUAGES, lang_key); nlang = _ent(LANGUAGES, neighbor_lang_key)
    where = (
        _c_country_base_lines("?country")
        + [f"?country wdt:P30 wd:{cont['qid']} .", f"?country wdt:P37 wd:{lang['qid']} .",
           f"?site wdt:P17 ?country .", f"?site wdt:P1435 wd:{Q_UNESCO_WHS} .",
           "?country wdt:P36 ?capital .", "?capital wdt:P1082 ?capitalPopulation .", f"FILTER(?capitalPopulation >= {int(cap_min)})",
           "?country wdt:P47 ?neighbor ."]
        + _c_current_country_lines("?neighbor", "neighbor")
        + [f"?neighbor wdt:P37 wd:{nlang['qid']} .", "FILTER(?neighbor != ?country)"]
        + _nobel_lines("?country")
    )
    return _base_spec(
        "L5", "countries_l5_nobel_whs_capital_border_language", "hard_people_border_heritage_bridge",
        {"continent": cont["en"], "official_language": lang["en"], "has_unesco_world_heritage_site": True, "has_nobel_prize_laureate_citizen": True, "capital_population_min": int(cap_min), "bordering_country_official_language": nlang["en"]}, where,
        f"{_query_head_ru()} в регионе «{cont['ru']}», где официальный язык — «{lang['ru']}», есть объект Всемирного наследия ЮНЕСКО, есть гражданин — лауреат Нобелевской премии, столица с населением не меньше {_fmt_num_ru(cap_min)} человек и граница со страной, где официальный язык — «{nlang['ru']}».",
        f"{_query_head_en()} in {cont['en']} whose official language is {lang['en']}, that have a UNESCO World Heritage Site, have a citizen who is a Nobel Prize laureate, have a capital with a population of at least {_fmt_num_en(cap_min)}, and border a country whose official language is {nlang['en']}.",
        "country <- Nobel laureate citizen; country <- UNESCO site; country -> capital population; country -> border country -> language",
        {"continent_qid": cont["qid"], "official_language_qid": lang["qid"], "bordering_country_official_language_qid": nlang["qid"], "nobel_prize_award_qids": NOBEL_PRIZE_QIDS, "unesco_world_heritage_site_qid": Q_UNESCO_WHS},
        {"continent": "P30", "official_language": "P37", "citizen_nobel_award": "inverse P27 + P166", "heritage_site_in_country": COUNTRY_PROPERTY_PATHS["heritage_site_in_country"], "capital_population": COUNTRY_PROPERTY_PATHS["capital_population"], "bordering_country_official_language": "P47 + P37"},
    )


def spec_l5_conflict_whs_capital_neighbor_org(cont_key: str, conflict_key: str, cap_min: int, neighbor_org_key: str) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); war = CONFLICTS[conflict_key]; org = _ent(ORGANIZATIONS, neighbor_org_key)
    where = (
        _c_country_base_lines("?country")
        + [f"?country wdt:P30 wd:{cont['qid']} .",
           f"?site wdt:P17 ?country .", f"?site wdt:P1435 wd:{Q_UNESCO_WHS} .",
           "?country wdt:P36 ?capital .", "?capital wdt:P1082 ?capitalPopulation .", f"FILTER(?capitalPopulation >= {int(cap_min)})",
           "?country wdt:P47 ?neighbor ."]
        + _conflict_participation_lines("?country", war["qid"])
        + _c_current_country_lines("?neighbor", "neighbor")
        + _c_current_member_lines("?neighbor", org["qid"], "neighborOrg")
        + ["FILTER(?neighbor != ?country)"]
    )
    return _base_spec(
        "L5", "countries_l5_conflict_whs_capital_neighbor_org", "hard_event_border_heritage_bridge",
        {"continent": cont["en"], "participated_in_conflict": war["en"], "has_unesco_world_heritage_site": True, "capital_population_min": int(cap_min), "bordering_country_member_of": org["en"]}, where,
        f"{_query_head_ru()} в регионе «{cont['ru']}», которые участвовали в событии «{war['ru']}», имеют объект Всемирного наследия ЮНЕСКО, столицу с населением не меньше {_fmt_num_ru(cap_min)} человек и границу со страной, которая сейчас входит в {_org_ru_text(org)}.",
        f"{_query_head_en()} in {cont['en']} that participated in {war['en']}, have a UNESCO World Heritage Site, have a capital with a population of at least {_fmt_num_en(cap_min)}, and border a current member of {_org_en_text(org)}.",
        "country -> conflict participation; country <- UNESCO site; country -> capital population; country -> border country -> current organization membership",
        {"continent_qid": cont["qid"], "conflict_qid": war["qid"], "bordering_country_member_of_qid": org["qid"], "unesco_world_heritage_site_qid": Q_UNESCO_WHS},
        {"continent": "P30", "participated_in_conflict": "P1344", "heritage_site_in_country": COUNTRY_PROPERTY_PATHS["heritage_site_in_country"], "capital_population": COUNTRY_PROPERTY_PATHS["capital_population"], "bordering_country_member_of_current": "P47 + current P463"},
    )


def spec_l5_olympic_whs_member_city_language(org_key: str, lang_key: str, city_min: int) -> Dict[str, Any]:
    org = _ent(ORGANIZATIONS, org_key); lang = _ent(LANGUAGES, lang_key)
    where = (
        _c_country_base_lines("?country")
        + _c_current_member_lines("?country", org["qid"], "org")
        + [f"?country wdt:P37 wd:{lang['qid']} .", f"?site wdt:P17 ?country .", f"?site wdt:P1435 wd:{Q_UNESCO_WHS} .",
           f"?city wdt:P31/wdt:P279* wd:{Q_CITY} .", "?city wdt:P17 ?country .", "?city wdt:P1082 ?cityPopulation .", f"FILTER(?cityPopulation >= {int(city_min)})"]
        + _olympic_medalist_lines("?country")
    )
    return _base_spec(
        "L5", "countries_l5_olympic_whs_member_city_language", "hard_sports_org_heritage_bridge",
        {"member_of": org["en"], "official_language": lang["en"], "has_unesco_world_heritage_site": True, "has_olympic_medalist_citizen": True, "has_city_population_min": int(city_min)}, where,
        f"{_query_head_ru()}, которые сейчас входят в {_org_ru_text(org)}, где официальный язык — «{lang['ru']}», есть объект Всемирного наследия ЮНЕСКО, гражданин — призёр Олимпийских игр и город с населением не меньше {_fmt_num_ru(city_min)} человек.",
        f"{_query_head_en()} that are current members of {_org_en_text(org)}, whose official language is {lang['en']}, that have a UNESCO World Heritage Site, have a citizen who is an Olympic medalist, and have a city with a population of at least {_fmt_num_en(city_min)}.",
        "country -> current membership; country <- Olympic medalist citizen; country <- UNESCO site; country <- large city; plus language",
        {"member_of_qid": org["qid"], "official_language_qid": lang["qid"], "olympic_medal_award_qids": OLYMPIC_MEDAL_QIDS, "unesco_world_heritage_site_qid": Q_UNESCO_WHS},
        {"member_of_current": COUNTRY_PROPERTY_PATHS["member_of_current"], "official_language": "P37", "citizen_olympic_medal": "inverse P27 + P166", "heritage_site_in_country": COUNTRY_PROPERTY_PATHS["heritage_site_in_country"], "large_city_in_country": COUNTRY_PROPERTY_PATHS["large_city_in_country"]},
    )


def spec_l5_hosted_event_nobel_capital_language(cont_key: str, event_key: str, lang_key: str, cap_min: int) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); ev = HOSTED_EVENT_TYPES[event_key]; lang = _ent(LANGUAGES, lang_key)
    where = (_c_country_base_lines("?country") + [f"?country wdt:P30 wd:{cont['qid']} .", f"?country wdt:P37 wd:{lang['qid']} .", "?country wdt:P36 ?capital .", "?capital wdt:P1082 ?capitalPopulation .", f"FILTER(?capitalPopulation >= {int(cap_min)})"] + _hosted_event_lines("?country", ev["qid"]) + _nobel_lines("?country"))
    return _base_spec(
        "L5", "countries_l5_hosted_event_nobel_capital_language", "hard_event_people_capital_bridge",
        {"continent": cont["en"], "official_language": lang["en"], "hosted_event_type": ev["en"], "has_nobel_prize_laureate_citizen": True, "capital_population_min": int(cap_min)}, where,
        f"{_query_head_ru()} в регионе «{cont['ru']}», где официальный язык — «{lang['ru']}», которые принимали событие типа «{ev['ru']}», имеют гражданина — лауреата Нобелевской премии и столицу с населением не меньше {_fmt_num_ru(cap_min)} человек.",
        f"{_query_head_en()} in {cont['en']} whose official language is {lang['en']}, that have hosted an event of type {ev['en']}, have a citizen who is a Nobel Prize laureate, and have a capital with a population of at least {_fmt_num_en(cap_min)}.",
        "country <- hosted event; country <- Nobel laureate citizen; country -> capital population; plus continent/language",
        {"continent_qid": cont["qid"], "official_language_qid": lang["qid"], "hosted_event_type_qid": ev["qid"], "nobel_prize_award_qids": NOBEL_PRIZE_QIDS},
        {"continent": "P30", "official_language": "P37", "hosted_event_type": "inverse P17 + P31/P279*", "citizen_nobel_award": "inverse P27 + P166", "capital_population": COUNTRY_PROPERTY_PATHS["capital_population"]},
    )


def spec_l5_capital_river_whs_nobel_border_org(cont_key: str, lang_key: str, neighbor_org_key: str) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); lang = _ent(LANGUAGES, lang_key); org = _ent(ORGANIZATIONS, neighbor_org_key)
    where = (_c_country_base_lines("?country") + [f"?country wdt:P30 wd:{cont['qid']} .", f"?country wdt:P37 wd:{lang['qid']} .", f"?site wdt:P17 ?country .", f"?site wdt:P1435 wd:{Q_UNESCO_WHS} .", "?country wdt:P47 ?neighbor ."] + _capital_on_river_lines("?country") + _nobel_lines("?country") + _c_current_country_lines("?neighbor", "neighbor") + _c_current_member_lines("?neighbor", org["qid"], "neighborOrg") + ["FILTER(?neighbor != ?country)"])
    return _base_spec(
        "L5", "countries_l5_capital_river_whs_nobel_border_org", "hard_geography_people_border_bridge",
        {"continent": cont["en"], "official_language": lang["en"], "has_unesco_world_heritage_site": True, "capital_located_on_river": True, "has_nobel_prize_laureate_citizen": True, "bordering_country_member_of": org["en"]}, where,
        f"{_query_head_ru()} в регионе «{cont['ru']}», где официальный язык — «{lang['ru']}», есть объект Всемирного наследия ЮНЕСКО, столица расположена на реке или у реки, есть гражданин — лауреат Нобелевской премии и граница со страной, которая сейчас входит в {_org_ru_text(org)}.",
        f"{_query_head_en()} in {cont['en']} whose official language is {lang['en']}, that have a UNESCO World Heritage Site, whose capital is located on or next to a river, have a citizen who is a Nobel Prize laureate, and border a current member of {_org_en_text(org)}.",
        "country -> capital -> river; country <- UNESCO site; country <- Nobel citizen; country -> border country -> current membership",
        {"continent_qid": cont["qid"], "official_language_qid": lang["qid"], "river_qid": Q_RIVER, "bordering_country_member_of_qid": org["qid"], "nobel_prize_award_qids": NOBEL_PRIZE_QIDS, "unesco_world_heritage_site_qid": Q_UNESCO_WHS},
        {"continent": "P30", "official_language": "P37", "capital_on_river": "P36/P206 + river class", "heritage_site_in_country": COUNTRY_PROPERTY_PATHS["heritage_site_in_country"], "citizen_nobel_award": "inverse P27 + P166", "bordering_country_member_of_current": "P47 + current P463"},
    )


def spec_l4_highest_point_whs_language(cont_key: str, lang_key: str, min_elevation_m: int) -> Dict[str, Any]:
    cont = _ent(CONTINENTS, cont_key); lang = _ent(LANGUAGES, lang_key)
    where = (_c_country_base_lines("?country") + [f"?country wdt:P30 wd:{cont['qid']} .", f"?country wdt:P37 wd:{lang['qid']} .", f"?site wdt:P17 ?country .", f"?site wdt:P1435 wd:{Q_UNESCO_WHS} ."] + _highest_point_lines("?country", min_elevation_m))
    return _base_spec(
        "L4", "countries_l4_highest_point_whs_language", "geography_heritage_bridge",
        {"continent": cont["en"], "official_language": lang["en"], "has_unesco_world_heritage_site": True, "highest_point_elevation_min_m": int(min_elevation_m)}, where,
        f"{_query_head_ru()} в регионе «{cont['ru']}», где официальный язык — «{lang['ru']}», есть объект Всемирного наследия ЮНЕСКО, а высшая точка имеет высоту не меньше {_fmt_num_ru(min_elevation_m)} метров.",
        f"{_query_head_en()} in {cont['en']} whose official language is {lang['en']}, that have a UNESCO World Heritage Site, and whose highest point has an elevation of at least {_fmt_num_en(min_elevation_m)} meters.",
        "country -> highest point -> elevation; country <- UNESCO site; plus continent/language",
        {"continent_qid": cont["qid"], "official_language_qid": lang["qid"], "unesco_world_heritage_site_qid": Q_UNESCO_WHS},
        {"continent": "P30", "official_language": "P37", "heritage_site_in_country": COUNTRY_PROPERTY_PATHS["heritage_site_in_country"], "highest_point_elevation": "P610/P2044"},
    )


def _add_v44_rich_specs(by_level: Dict[str, List[Dict[str, Any]]]) -> None:
    continents_all = ["Africa", "Europe", "Asia", "North America", "South America", "Oceania"]
    language_by_cont = {
        "Africa": ["French", "Arabic", "English", "Portuguese"],
        "Europe": ["English", "German", "French", "Spanish", "Italian"],
        "Asia": ["Arabic", "English", "French", "Russian"],
        "North America": ["English", "Spanish", "French"],
        "South America": ["Spanish", "Portuguese", "English"],
        "Oceania": ["English", "French"],
    }
    useful_currencies = ["euro", "United States dollar", "CFA franc BCEAO", "CFA franc BEAC", "East Caribbean dollar", "pound sterling"]
    useful_orgs = ["European Union", "NATO", "OECD", "Commonwealth of Nations", "African Union", "Arab League", "United Nations"]

    # L3: add broader event/people/geography bridges across continents.
    for cont in continents_all:
        for lang in language_by_cont.get(cont, []):
            if lang in LANGUAGES:
                by_level["L3"].append(spec_l3_continent_language_nobel(cont, lang))
                by_level["L3"].append(spec_l3_continent_language_olympic_medalist(cont, lang))
        for event in HOSTED_EVENT_TYPES:
            by_level["L3"].append(spec_l3_continent_hosted_event(cont, event))
        for conflict in CONFLICTS:
            by_level["L3"].append(spec_l3_continent_conflict(cont, conflict))

    # L4: high-recall hard rows; positive-only constraints.
    for cont in continents_all:
        for lang in language_by_cont.get(cont, []):
            if lang in LANGUAGES:
                for cap in [100_000, 250_000, 500_000, 1_000_000]:
                    by_level["L4"].append(spec_l4_language_whs_nobel_capital(cont, lang, cap))
                by_level["L4"].append(spec_l4_capital_river_whs_language(cont, lang))
                for elev in [1000, 2000, 3000]:
                    by_level["L4"].append(spec_l4_highest_point_whs_language(cont, lang, elev))
                for event in HOSTED_EVENT_TYPES:
                    by_level["L4"].append(spec_l4_hosted_event_language_capital(cont, event, lang, 100_000))
                    by_level["L4"].append(spec_l4_hosted_event_language_capital(cont, event, lang, 500_000))
        for curr in useful_currencies:
            if curr in CURRENCIES:
                for city_min in [100_000, 500_000, 1_000_000]:
                    by_level["L4"].append(spec_l4_currency_whs_olympic_city(cont, curr, city_min))
        for conflict in CONFLICTS:
            for cap in [100_000, 500_000, 1_000_000]:
                by_level["L4"].append(spec_l4_conflict_whs_capital(cont, conflict, cap))

    # L5: richer multi-bridge rows. Many are selective, so generate a large queue.
    neighbor_langs = ["English", "French", "Spanish", "Arabic", "German", "Portuguese", "Russian"]
    for cont in continents_all:
        for lang in language_by_cont.get(cont, []):
            if lang not in LANGUAGES:
                continue
            for nlang in neighbor_langs:
                if nlang in LANGUAGES:
                    for cap in [100_000, 250_000, 500_000, 1_000_000]:
                        by_level["L5"].append(spec_l5_nobel_whs_capital_border_language(cont, lang, cap, nlang))
            for org in useful_orgs:
                if org in ORGANIZATIONS:
                    by_level["L5"].append(spec_l5_capital_river_whs_nobel_border_org(cont, lang, org))
            for event in HOSTED_EVENT_TYPES:
                for cap in [100_000, 500_000, 1_000_000]:
                    by_level["L5"].append(spec_l5_hosted_event_nobel_capital_language(cont, event, lang, cap))
        for conflict in CONFLICTS:
            for org in useful_orgs:
                if org in ORGANIZATIONS:
                    for cap in [100_000, 500_000, 1_000_000]:
                        by_level["L5"].append(spec_l5_conflict_whs_capital_neighbor_org(cont, conflict, cap, org))

    for org in useful_orgs:
        if org not in ORGANIZATIONS:
            continue
        for lang in neighbor_langs:
            if lang in LANGUAGES:
                for city_min in [100_000, 500_000, 1_000_000]:
                    by_level["L5"].append(spec_l5_olympic_whs_member_city_language(org, lang, city_min))


# Keep v43 builder, then extend its output and dedupe specs again.
_countries_build_candidate_specs_v43 = build_countries_candidate_specs

def build_countries_candidate_specs(seed: int = COUNTRIES_RANDOM_SEED) -> Dict[str, List[Dict[str, Any]]]:
    by_level = _countries_build_candidate_specs_v43(seed)
    _add_v44_rich_specs(by_level)

    # Dedupe specs by clean constraints + template_id; keep curated/v43 rows first, then v44 rows.
    for lvl, specs in by_level.items():
        seen = set()
        uniq = []
        for s in specs:
            if _countries_spec_veto_reason(s):
                continue
            sig = _c_signature(s)
            if sig in seen:
                continue
            seen.add(sig)
            uniq.append(s)
        # Deterministic interleaving: keep order but avoid Africa-only streaks by stable round-robin over primary constraint.
        buckets = defaultdict(list)
        for s in uniq:
            c = s.get("constraints", {}) or {}
            key = c.get("continent") or c.get("member_of") or c.get("official_language") or c.get("currency") or "global"
            buckets[str(key)].append(s)
        ordered = []
        keys = list(buckets.keys())
        # Put non-Africa buckets before Africa in each pass to reduce geographic skew in early accepted records.
        keys.sort(key=lambda k: (k == "Africa" or k == "African Union", k))
        while any(buckets.values()):
            for k in keys:
                if buckets[k]:
                    ordered.append(buckets[k].pop(0))
        by_level[lvl] = ordered[:COUNTRIES_MAX_CANDIDATES_PER_LEVEL]
    return by_level

COUNTRIES_CANDIDATE_QUEUES = build_countries_candidate_specs(COUNTRIES_RANDOM_SEED)
print("v44 target_per_level:", COUNTRIES_TARGET_PER_LEVEL)
for lvl, q in COUNTRIES_CANDIDATE_QUEUES.items():
    prim = Counter((s.get("constraints", {}) or {}).get("continent") or (s.get("constraints", {}) or {}).get("member_of") or "global" for s in q)
    print(lvl, "candidates:", len(q), "top_primary:", prim.most_common(8))


v44 target_per_level: {'L1': 18, 'L2': 22, 'L3': 35, 'L4': 55, 'L5': 55}
L1 candidates: 167 top_primary: [('Arab League', 12), ('Commonwealth of Nations', 12), ('Europe', 12), ('European Union', 12), ('Mercosur', 12), ('NATO', 12), ('North America', 12), ('OECD', 12)]
L2 candidates: 1244 top_primary: [('Africa', 147), ('Europe', 145), ('North America', 145), ('Oceania', 144), ('South America', 144), ('Asia', 131), ('Commonwealth of Nations', 50), ('OECD', 49)]
L3 candidates: 470 top_primary: [('Europe', 71), ('Africa', 71), ('North America', 68), ('South America', 68), ('Oceania', 66), ('Asia', 58), ('Arab League', 9), ('Commonwealth of Nations', 9)]
L4 candidates: 1131 top_primary: [('Africa', 194), ('Europe', 188), ('North America', 172), ('South America', 172), ('Asia', 168), ('Oceania', 159), ('Commonwealth of Nations', 12), ('African Union', 12)]
L5 candidates: 2121 top_primary: [('Africa', 330), ('Europe', 323), ('North America', 277), ('South America', 275), ('Asia', 261), ('Oce

In [8]:
# ============================================================
# Quality checks, generation, resume, and incremental save
# ============================================================

def countries_normalize_record(r: Dict[str, Any]) -> Dict[str, Any]:
    # Ensure the exact same top-level schema style as cinema/people JSONLs.
    out = dict(r)
    out.setdefault("query_text_en", "")
    out.setdefault("gold_answer_labels_en", [])
    out.setdefault("is_advanced", out.get("complexity") in {"L3", "L4", "L5"})
    out.setdefault("template_id", "countries_unknown")
    out.setdefault("template_family", "countries")
    out.setdefault("gold_truncated", False)
    out.setdefault("ask_validator_sparql", "")
    out.setdefault("local_validator", {
        "type": "none_wdqs_only",
        "source": "Wikidata Query Service",
        "match_key": "wikidata_qid",
        "applies_after": "ask_validator_sparql",
        "filters": out.get("constraints", {}),
        "label_matching_used": False,
        "note": "Use ask_validator_sparql for all Wikidata constraints; no external local validator is required for countries-domain examples.",
    })
    out.setdefault("gold_collection_meta", {})
    out.setdefault("gold_answer_imdb_ids", [])
    out.setdefault("gold_answer_imdb_titles", [])
    return out


def countries_record_quality_problems(r: Dict[str, Any]) -> List[str]:
    problems: List[str] = []
    required = ["id", "domain", "complexity", "query_text_ru", "query_text_en", "constraints", "requested_count", "gold_answer_qids", "gold_answer_labels_ru", "gold_answer_labels_en", "sparql_query", "ask_validator_sparql", "local_validator", "gold_collection_meta"]
    for k in required:
        if k not in r:
            problems.append(f"missing_{k}")
    if r.get("domain") != COUNTRIES_DOMAIN:
        problems.append("wrong_domain")
    qids = r.get("gold_answer_qids") or []
    ru = r.get("gold_answer_labels_ru") or []
    en = r.get("gold_answer_labels_en") or []
    if len(qids) < int(r.get("requested_count", 5)):
        problems.append("gold_lt_requested")
    if len(qids) != len(ru) or len(qids) != len(en):
        problems.append("gold_label_length_mismatch")
    if len(qids) != len(set(qids)):
        problems.append("duplicate_gold_qids")
    if r.get("gold_truncated"):
        problems.append("gold_truncated_true")
    lv = r.get("local_validator") or {}
    if lv.get("filters") != r.get("constraints"):
        problems.append("local_validator_filters_do_not_match_constraints")
    sparql_text = str(r.get("sparql_query") or "")
    ask_text = str(r.get("ask_validator_sparql") or "")
    if not sparql_text or "SELECT" not in sparql_text:
        problems.append("missing_select_sparql")
    if not ask_text or "ASK" not in ask_text:
        problems.append("missing_ask_validator")
    # v36+ strictness: do not resume old rows generated with broad `country` subclass matching,
    # because it leaked micronations, territories and constituent countries into golds.
    if "wdt:P31 wd:Q3624078" not in sparql_text or "wdt:P31 wd:Q3624078" not in ask_text:
        problems.append("non_strict_sovereign_state_filter")
    c = r.get("constraints", {}) or {}
    if "not_member_of" in c:
        problems.append("unsafe_negative_membership_not_member_of")
    if c.get("continent") == "Asia" and c.get("official_language") == "Arabic" and c.get("member_of") != "Arab League":
        problems.append("unstable_generic_asia_arabic_official_language")
    false_exclusions = set(COUNTRIES_FALSE_OFFICIAL_LANGUAGE_QID_EXCLUSIONS.get(str(c.get("official_language", "")), []))
    if false_exclusions and any(qid in false_exclusions for qid in qids):
        problems.append("known_false_official_language_gold")
    return problems


def countries_validate_records(records: Sequence[Dict[str, Any]]) -> Dict[str, Any]:
    ids = [r.get("id") for r in records]
    queries = [_c_query_signature(r) for r in records]
    constraints = [json.dumps(r.get("constraints", {}), ensure_ascii=False, sort_keys=True) for r in records]
    by_level = Counter(r.get("complexity") for r in records)
    problem_rows = []
    for r in records:
        probs = countries_record_quality_problems(r)
        if probs:
            problem_rows.append({"id": r.get("id"), "problems": probs})
    return {
        "total": len(records),
        "by_level": dict(sorted(by_level.items())),
        "duplicate_ids": [x for x, n in Counter(ids).items() if n > 1],
        "duplicate_query_text_ru_count": sum(1 for _, n in Counter(queries).items() if n > 1),
        "duplicate_constraints_count": sum(1 for _, n in Counter(constraints).items() if n > 1),
        "gold_lt_requested_count": sum(1 for r in records if _c_gold_count(r) < int(r.get("requested_count", 5))),
        "gold_count_min": min([_c_gold_count(r) for r in records], default=0),
        "gold_count_max": max([_c_gold_count(r) for r in records], default=0),
        "records_with_quality_problems": problem_rows[:100],
    }



def _countries_gold_set(r: Dict[str, Any]) -> set:
    return set(r.get("gold_answer_qids") or [])


def countries_gold_overlap_reason(candidate: Dict[str, Any], previous_records: Sequence[Dict[str, Any]]) -> Optional[Dict[str, Any]]:
    """Return a duplicate reason if candidate's answer set is too similar to a prior record.

    v41 change: countries L2/L3 are often legitimate subsets of L1. Reject exact/near-exact
    answer-set duplicates across any levels, but apply containment-based rejection only when
    the two answer sets are nearly the same size. This prevents L2 from getting starved by
    broad L1 records while still removing cases like Africa+French == African Union+French.
    """
    gold = _countries_gold_set(candidate)
    if len(gold) < int(candidate.get("requested_count", 5)):
        return None
    cand_level = candidate.get("complexity")
    cand_template = candidate.get("template_id")
    for prev in previous_records:
        pgold = _countries_gold_set(prev)
        if len(pgold) < int(prev.get("requested_count", 5)):
            continue
        inter = len(gold & pgold)
        if inter < int(candidate.get("requested_count", 5)):
            continue
        union = len(gold | pgold)
        jaccard = inter / union if union else 0.0
        containment = inter / min(len(gold), len(pgold)) if min(len(gold), len(pgold)) else 0.0
        size_ratio = min(len(gold), len(pgold)) / max(len(gold), len(pgold)) if max(len(gold), len(pgold)) else 0.0

        # Always reject identical / near-identical answer sets.
        is_near_exact = jaccard >= COUNTRIES_DEDUP_GOLD_JACCARD_THRESHOLD

        # Reject subset-style duplicates only when the smaller set is almost the same size.
        # This keeps L2 from being blocked merely because it is a useful narrowed version of L1.
        is_containment_duplicate = (
            containment >= COUNTRIES_DEDUP_GOLD_CONTAINMENT_THRESHOLD
            and size_ratio >= COUNTRIES_DEDUP_GOLD_SIZE_RATIO_THRESHOLD
        )

        if is_near_exact or is_containment_duplicate:
            return {
                "reason": "gold_overlap_duplicate",
                "with_id": prev.get("id"),
                "with_template_id": prev.get("template_id"),
                "intersection": inter,
                "candidate_gold_count": len(gold),
                "previous_gold_count": len(pgold),
                "jaccard": round(jaccard, 4),
                "containment": round(containment, 4),
                "size_ratio": round(size_ratio, 4),
                "candidate_level": cand_level,
                "previous_level": prev.get("complexity"),
                "candidate_template_id": cand_template,
            }
    return None

def _countries_next_index_by_level(records: Sequence[Dict[str, Any]], references: Sequence[Dict[str, Any]]) -> Dict[str, int]:
    max_by_level = defaultdict(int)
    for r in list(records) + list(references):
        lvl = r.get("complexity")
        rid = str(r.get("id", ""))
        m = re.search(r"countries_l(\d+)_(\d+)$", rid)
        if m:
            idx = int(m.group(2))
            max_by_level[f"L{m.group(1)}"] = max(max_by_level[f"L{m.group(1)}"], idx)
        elif lvl in COUNTRIES_TARGET_PER_LEVEL:
            max_by_level[lvl] = max(max_by_level[lvl], 0)
    return {lvl: max_by_level[lvl] + 1 for lvl in COUNTRIES_TARGET_PER_LEVEL}


def generate_countries_dataset() -> List[Dict[str, Any]]:
    records_existing, bad_existing = _c_read_jsonl(COUNTRIES_OUTPUT_PATH)
    if bad_existing:
        print("bad existing lines:", bad_existing[:5])
    records: List[Dict[str, Any]] = []
    skipped_existing: List[Dict[str, Any]] = []
    for r in records_existing:
        rr = countries_normalize_record(r)
        probs = countries_record_quality_problems(rr)
        if probs:
            skipped_existing.append({"id": rr.get("id"), "problems": probs, "query_text_ru": rr.get("query_text_ru")})
            continue
        overlap = countries_gold_overlap_reason(rr, records)
        if overlap:
            skipped_existing.append({"id": rr.get("id"), "problems": [overlap["reason"]], "overlap": overlap, "query_text_ru": rr.get("query_text_ru")})
            continue
        records.append(rr)
    if skipped_existing:
        print("dropping existing low-quality/duplicate rows before resume:", len(skipped_existing))
        _c_rewrite_jsonl(COUNTRIES_OUTPUT_PATH, records)

    # By default, do NOT use previous partial countries files as hard references.
    # They are often generated by earlier notebook versions and can block nearly all
    # candidates through gold-overlap dedup, making generation look "stuck".
    # Existing rows in the active output file are still used for resume/dedup.
    ref_records: List[Dict[str, Any]] = []
    if COUNTRIES_USE_REFERENCE_FILES_FOR_DEDUP:
        ref_paths = [p for p in _c_find_reference_jsonl_paths() if p.resolve() != COUNTRIES_OUTPUT_PATH.resolve()]
        for p in ref_paths:
            rows, bad = _c_read_jsonl(p)
            for row in rows:
                rr = countries_normalize_record(row)
                if not countries_record_quality_problems(rr):
                    ref_records.append(rr)
    print("existing output records:", len(records))
    print("reference records for exact dedup:", len(ref_records), "(disabled for speed unless COUNTRIES_USE_REFERENCE_FILES_FOR_DEDUP=True)")

    existing_sigs = {_c_signature(r) for r in records + ref_records}
    existing_queries = {_c_query_signature(r) for r in records + ref_records if _c_query_signature(r)}
    next_idx = _countries_next_index_by_level(records, ref_records)
    accepted_by_level = Counter(r.get("complexity") for r in records)
    template_counts = Counter((r.get("complexity"), r.get("template_id")) for r in records)
    primary_counts = Counter()
    for r in records:
        c = r.get("constraints", {}) or {}
        primary = c.get("continent") or c.get("member_of") or c.get("official_language") or c.get("currency")
        if primary:
            primary_counts[(r.get("complexity"), str(primary))] += 1

    audit = {
        "created_at": _c_now(),
        "output_path": str(COUNTRIES_OUTPUT_PATH),
        "target_per_level": COUNTRIES_TARGET_PER_LEVEL,
        "skipped_existing": skipped_existing,
        "skipped_candidates": [],
        "accepted": [],
    }

    for lvl in ["L1", "L2", "L3", "L4", "L5"]:
        target = int(COUNTRIES_TARGET_PER_LEVEL.get(lvl, 0))
        need = max(0, target - accepted_by_level.get(lvl, 0))
        if need <= 0:
            print(lvl, "already complete", accepted_by_level.get(lvl, 0), "/", target)
            continue
        queue = list(COUNTRIES_CANDIDATE_QUEUES.get(lvl, []))
        print(lvl, "need", need, "candidates", len(queue), "(progress bar counts accepted records; tried candidates are in postfix)")
        pbar = tqdm(total=need, desc=f"countries {lvl}", unit="record")
        tried_candidates = 0
        last_accept_tried = 0
        stagnation_limit = int(COUNTRIES_STAGNATION_LIMIT_BY_LEVEL.get(lvl, 10**9))
        min_before_stagnation = int(COUNTRIES_MIN_ACCEPTED_BEFORE_STAGNATION.get(lvl, 10**9))
        for spec in queue:
            tried_candidates += 1
            if accepted_by_level.get(lvl, 0) >= target:
                break
            if (accepted_by_level.get(lvl, 0) >= min_before_stagnation and
                tried_candidates - last_accept_tried >= stagnation_limit):
                audit["skipped_candidates"].append({
                    "level": lvl,
                    "reason": "adaptive_stagnation_stop",
                    "accepted_total": accepted_by_level.get(lvl, 0),
                    "tried": tried_candidates,
                    "last_accept_tried": last_accept_tried,
                    "stagnation_limit": stagnation_limit,
                })
                print("adaptive stop:", lvl, "accepted", accepted_by_level.get(lvl, 0), "tried", tried_candidates)
                break
            if tried_candidates % 10 == 0:
                pbar.set_postfix({"accepted_total": accepted_by_level.get(lvl, 0), "tried": tried_candidates, "candidates": len(queue)})
            if _c_signature(spec) in existing_sigs:
                audit["skipped_candidates"].append({"level": lvl, "reason": "duplicate_signature", "template_id": spec.get("template_id"), "constraints": spec.get("constraints")})
                continue
            if _c_query_signature(spec) in existing_queries:
                audit["skipped_candidates"].append({"level": lvl, "reason": "duplicate_query_text", "template_id": spec.get("template_id"), "constraints": spec.get("constraints")})
                continue
            veto_reason = _countries_spec_veto_reason(spec)
            if veto_reason:
                audit["skipped_candidates"].append({"level": lvl, "reason": veto_reason, "template_id": spec.get("template_id"), "constraints": spec.get("constraints")})
                continue
            if template_counts[(lvl, spec.get("template_id"))] >= COUNTRIES_MAX_SAME_TEMPLATE_PER_LEVEL.get(lvl, 999):
                audit["skipped_candidates"].append({"level": lvl, "reason": "template_quota", "template_id": spec.get("template_id")})
                continue
            c = spec.get("constraints", {}) or {}
            primary = c.get("continent") or c.get("member_of") or c.get("official_language") or c.get("currency")
            primary_quota = (COUNTRIES_MAX_SAME_PRIMARY_CONSTRAINT_PER_LEVEL.get(lvl, 999)
                             if isinstance(COUNTRIES_MAX_SAME_PRIMARY_CONSTRAINT_PER_LEVEL, dict)
                             else COUNTRIES_MAX_SAME_PRIMARY_CONSTRAINT_PER_LEVEL)
            if primary and primary_counts[(lvl, str(primary))] >= primary_quota:
                audit["skipped_candidates"].append({"level": lvl, "reason": "primary_constraint_quota", "primary": primary, "quota": primary_quota})
                continue

            spec = dict(spec)
            spec["id"] = f"countries_{lvl.lower()}_{next_idx[lvl]:04d}"
            rec, status = _c_collect_gold(spec)
            if rec is None:
                status.update({"level": lvl, "constraints": spec.get("constraints"), "query_text_ru": spec.get("query_text_ru")})
                audit["skipped_candidates"].append(status)
                continue

            rec = countries_normalize_record(rec)
            probs = countries_record_quality_problems(rec)
            if probs:
                audit["skipped_candidates"].append({"level": lvl, "reason": "quality_problems_after_build", "problems": probs, "id": rec.get("id")})
                continue
            overlap = countries_gold_overlap_reason(rec, records)
            if overlap:
                audit["skipped_candidates"].append({"level": lvl, **overlap, "id": rec.get("id"), "template_id": rec.get("template_id"), "constraints": rec.get("constraints")})
                continue

            _c_append_jsonl(COUNTRIES_OUTPUT_PATH, rec)
            last_accept_tried = tried_candidates
            records.append(rec)
            existing_sigs.add(_c_signature(rec))
            existing_queries.add(_c_query_signature(rec))
            next_idx[lvl] += 1
            accepted_by_level[lvl] += 1
            template_counts[(lvl, rec.get("template_id"))] += 1
            c = rec.get("constraints", {}) or {}
            primary = c.get("continent") or c.get("member_of") or c.get("official_language") or c.get("currency")
            if primary:
                primary_counts[(lvl, str(primary))] += 1
            audit["accepted"].append({"id": rec.get("id"), "level": lvl, "template_id": rec.get("template_id"), "gold_count": _c_gold_count(rec)})
            pbar.update(1)
            pbar.set_postfix({
                "accepted_total": accepted_by_level.get(lvl, 0),
                "tried": tried_candidates,
                "candidates": len(queue),
                "last_gold_count": _c_gold_count(rec),
            })

        pbar.close()
        if accepted_by_level.get(lvl, 0) < target:
            print("WARNING:", lvl, "target not reached:", accepted_by_level.get(lvl, 0), "/", target)

    stats = countries_validate_records(records)
    audit["final_stats"] = stats
    with open(COUNTRIES_AUDIT_PATH, "w", encoding="utf-8") as f:
        json.dump(audit, f, ensure_ascii=False, indent=2, default=_c_json_default)
    with open(COUNTRIES_CHECKPOINT_PATH, "w", encoding="utf-8") as f:
        json.dump({"next_index_by_level": next_idx, "accepted_by_level": dict(accepted_by_level)}, f, ensure_ascii=False, indent=2)

    print(json.dumps(stats, ensure_ascii=False, indent=2))
    return records


RUN_COUNTRIES_GENERATION = True
if RUN_COUNTRIES_GENERATION:
    countries_records = generate_countries_dataset()
else:
    countries_records, _ = _c_read_jsonl(COUNTRIES_OUTPUT_PATH)
    print(json.dumps(countries_validate_records(countries_records), ensure_ascii=False, indent=2))


existing output records: 77
reference records for exact dedup: 0 (disabled for speed unless COUNTRIES_USE_REFERENCE_FILES_FOR_DEDUP=True)
L1 need 4 candidates 167 (progress bar counts accepted records; tried candidates are in postfix)


countries L1:  25%|██▌       | 1/4 [00:00<00:00, 18.20record/s, accepted_total=15, tried=160, candidates=167]                    


L2 need 8 candidates 1244 (progress bar counts accepted records; tried candidates are in postfix)


countries L2:  38%|███▊      | 3/8 [00:00<00:00, 22.72record/s, accepted_total=17, tried=400, candidates=1244]                    


adaptive stop: L2 accepted 17 tried 403
L3 need 13 candidates 470 (progress bar counts accepted records; tried candidates are in postfix)


...

L4 need 37 candidates 1131 (progress bar counts accepted records; tried candidates are in postfix)


countries L4:  54%|█████▍    | 20/37 [35:12<29:55, 105.62s/record, accepted_total=38, tried=1130, candidates=1131]                  


L5 need 46 candidates 2121 (progress bar counts accepted records; tried candidates are in postfix)


countries L5:   2%|▏         | 1/46 [59:00<7:38:32, 611.40s/record, accepted_total=10, tried=1470, candidates=2121]                  

KeyboardInterrupt: 

In [ ]:
# ============================================================
# Preview and schema sanity check
# ============================================================
records, bad_lines = _c_read_jsonl(COUNTRIES_OUTPUT_PATH)
records = [countries_normalize_record(r) for r in records]
print("bad lines:", bad_lines[:5])
print(json.dumps(countries_validate_records(records), ensure_ascii=False, indent=2))

# Expected top-level key order, matching cinema/people style.
EXPECTED_COUNTRIES_KEYS = [
    "id", "domain", "complexity", "query_text_ru", "constraints", "requested_count",
    "gold_answer_qids", "gold_answer_labels_ru", "sparql_query", "created_at",
    "query_text_en", "gold_answer_labels_en", "is_advanced", "template_id", "template_family",
    "gold_truncated", "ask_validator_sparql", "local_validator", "gold_collection_meta",
    "gold_answer_imdb_ids", "gold_answer_imdb_titles",
]

key_mismatches = []
for r in records:
    if list(r.keys()) != EXPECTED_COUNTRIES_KEYS:
        key_mismatches.append({"id": r.get("id"), "keys": list(r.keys())})
print("schema key mismatches:", len(key_mismatches))
if key_mismatches:
    print(key_mismatches[:3])

for r in records[:5]:
    print("\n", r.get("id"), r.get("complexity"), r.get("template_id"), "gold=", _c_gold_count(r))
    print("RU:", r.get("query_text_ru"))
    print("EN:", r.get("query_text_en"))
    print("constraints:", json.dumps(r.get("constraints", {}), ensure_ascii=False))
